# colab_17 — scGPT CPT evals #1 & #2 (aggregated regime · seed 0)

The eval battery for the scGPT continued-pretraining checkpoint produced in colab_16. Same two
pre-committed evals the Geneformer arm was scored on (`docs/EVALUATION_CONTRACT.md`):

- **eval #1** — linear probe on **substate** (microglia homeostatic vs activated; astrocyte resting
  vs reactive; `intermediate` excluded), donor-held-out.
- **eval #2** — **APOE-carrier recovery** within microglia and within astrocytes (k-NN load-bearing,
  silhouette corroborating-only). The Stanton-core axis.

**What this notebook is answering.** colab_16 found scGPT's CPT drift REAL and *large*: 22.25x its
measured noise floor, and 234.5% / 149.3% (micro / astro) of the within-donor substate reference
distance recomputed in scGPT's own space — i.e. the embedding moved further than the distance between
biologically distinct cell states. Nothing so far says *where* it moved. If the evals below are null,
that drift is largely orthogonal to both the substate and the APOE axis; if they are not, this is the
first eval-recoverable CPT effect in the project.

**One extraction point, and why that is complete here.** The Geneformer notebooks had to read every
eval at two depths (`emb_layer=-1` and `emb_layer=0`) because that arm's loss gain sat downstream of
the pipeline readout, in the head and last encoder layer — even the deeper readout recovered only
part of it. scGPT has no such gap: the cell embedding is the `<cls>` position at the top of the
encoder, every LoRA target (`self_attn`, `linear1`, `linear2` across the encoder layers) is
**upstream** of it, and the `ExprDecoder` that produced the CPT loss is frozen and downstream. The
single readout therefore sees the complete adapted representation, so the "partial view" caveat that
qualified colab_12 / colab_15 does not apply to these numbers.

**What is new here: a measured noise floor for the eval metrics themselves.** scGPT randomly
subsamples the genes of any cell above its 1200-token context, and 86.34% of this substrate is above
it — so the same cell embeds to a slightly different vector on every pass, and a small balanced-accuracy
Δ cannot be told apart from re-embedding stochasticity by inspection. Under `MEASURE_FLOOR=True` the
frozen base and the merged adapter are each re-embedded three times and the **entire battery is run on
those passes too**, giving a null (base pass vs base pass, where the true effect is zero by
construction) and a spread on the real effect. This is affordable only because scGPT embeds this
substrate in ca. 3.5 min against Geneformer's ca. 1h43m. The contract's bands are **not** rewritten
from what comes back — reporting a measured null alongside a pre-registered band is not the same move
as retuning the band to fit it.

**Tiers.** `MEASURE_FLOOR=False` (committed default) scores the evals from embeddings already on
Drive: no scGPT install, no GPU, no compute units — select a **no-accelerator runtime** for it.
`MEASURE_FLOOR=True` needs a GPU and adds six embedding passes (ca. 3.5 min each). Computing the
passes is gated by the flag; *using* them is not, so once a GPU run has written them, later CPU-only
runs read the floor for free. Note the second, CPU-side cost of the floor tier: the battery then
scores eight variants instead of two, so the probe fits, k-NN fits and silhouette computations in
sections 7 and 8 take roughly four times as long as a bare tier-1 run.

**Ungated by design, for now.** A `meaningful`/`decisive` verdict here would still owe the contract's
detector #2 (forgetting), which exists only for the Geneformer aggregated checkpoint. There is no
scGPT forgetting probe yet; the audit trace records that gap machine-readably rather than only in
prose.

## 1 — Setup

### 1a — Run-control flags, Drive, repo, conditional scGPT install

Two live switches. `MEASURE_FLOOR` is the expensive one: leave it `False` and this notebook is a
CPU-only read of existing embeddings; flip it to `True` **and select a GPU runtime first**, or the
assert below stops the run rather than silently falling back to CPU.

In [1]:
import os, subprocess, sys
from google.colab import drive

# ------------------------------ the two live switches ------------------------------
SMOKE         = False   # plumbing rehearsal on a tiny subsample; never writes the audit trace
MEASURE_FLOOR = True   # False -> evals only, from embeddings already on Drive (CPU, 0 compute units)
                        # True  -> ALSO re-embed the frozen base and the merged adapter N times each
                        #          to measure the eval-metric noise floor (GPU, 6 passes)
# ----------------------------------------------------------------------------------

N_BASE_PASSES = 3       # repeat embeddings of the FROZEN base -> the null (true effect is zero)
N_CPT_PASSES  = 3       # repeat embeddings of the MERGED adapter -> spread on the real effect
SMOKE_CAP     = 40      # cells per (split x lineage x substate) group when SMOKE
SUFFIX        = "_SMOKE" if SMOKE else ""
RUN_TAG       = "seed0" # the colab_16 checkpoint these evals score

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print("Python:", sys.version.split()[0])
print("repo commit:", subprocess.run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"],
                                     capture_output=True, text=True).stdout.strip())

SCGPT_PIN  = "cebd6fae655b9c585a4807daa3ac31bb764f06b4"
MODEL_DIR  = os.path.join(DRIVE_ROOT, "scgpt_whole_human")
CKPT_FILES = ["vocab.json", "args.json", "best_model.pt"]

if MEASURE_FLOOR:
    # scGPT source only (--no-deps) at the same commit every existing embedding was produced under.
    # flash-attn stays deliberately absent -> PyTorch attention, which is what colab_10 and colab_16
    # embedded under; installing it here would make these passes a different code path.
    !pip install --no-deps "git+https://github.com/bowang-lab/scGPT.git@{SCGPT_PIN}"
    !pip install -r {REPO_PATH}/requirements_scgpt.txt
    # Colab's base image ships torchao 0.10.0, and peft 0.19.1's `is_torchao_available()` RAISES on
    # any version below 0.16.0 instead of returning False -- that kills the adapter reload in 5b.
    # Nothing in the scGPT stack imports torchao, so removing it makes the probe return False.
    !pip uninstall -y torchao

    # 5b reloads colab_16's adapter with PeftModel.from_pretrained + merge_and_unload. peft's
    # nn.MultiheadAttention merge behaviour is correctness-critical here (docs/ASSUMPTIONS.md) --
    # assert the version rather than assume it.
    import peft
    PEFT_PIN = "0.19.1"
    assert peft.__version__ == PEFT_PIN, (
        f"peft {peft.__version__} != pinned {PEFT_PIN}; the adapter reload/merge in 5b depends on "
        "its nn.MultiheadAttention behaviour -- do not proceed on a different version.")
    print("peft:", peft.__version__)

    def _have_ckpt(d):
        return all(os.path.exists(os.path.join(d, f)) for f in CKPT_FILES)

    if not _have_ckpt(MODEL_DIR):
        os.makedirs(MODEL_DIR, exist_ok=True)
        WHOLE_HUMAN_FOLDER = "1oWh_-ZRdhtoGQ2Fw24HP41FgLoomVo-y"   # scGPT README pretrained table
        !pip install -q gdown
        import gdown
        gdown.download_folder(id=WHOLE_HUMAN_FOLDER, output=MODEL_DIR, quiet=False, use_cookies=False)
        hits = [dp for dp, _, fs in os.walk(MODEL_DIR) if "best_model.pt" in fs]
        assert hits, f"best_model.pt not found under {MODEL_DIR} after download"
        MODEL_DIR = hits[0]
    assert _have_ckpt(MODEL_DIR), f"checkpoint incomplete in {MODEL_DIR}: need {CKPT_FILES}"

    import torch
    assert torch.cuda.is_available(), (
        "MEASURE_FLOOR=True needs a GPU runtime -- select one and re-run. (Running the six "
        "embedding passes on CPU is not viable at 142,588 cells.)")
    SCGPT_COMMIT = SCGPT_PIN
    print("scGPT commit:", SCGPT_COMMIT[:7], "| checkpoint:", MODEL_DIR, "| GPU:",
          torch.cuda.get_device_name(0))
else:
    # The evals read embeddings that are already on Drive: no FM, no checkpoint, no GPU. scanpy /
    # anndata / scikit-learn arrive with requirements_scgpt.txt in the branch above, so this branch
    # has to ask for them itself or 2a's `import scanpy` crashes on a stock Colab image.
    !pip install -q scanpy anndata scikit-learn
    # Adopted at 4a from the commit the existing embeddings were actually recorded under, rather
    # than asserted here from a value this run has no way to verify.
    SCGPT_COMMIT = None
    print("MEASURE_FLOOR=False -- evals only, from existing embeddings.")
    print("  no scGPT install, no checkpoint download, no GPU needed (pick a no-accelerator runtime)")

print(f"\nSMOKE={SMOKE} | MEASURE_FLOOR={MEASURE_FLOOR} | suffix={SUFFIX!r}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.13
repo commit: 899bfd338ed51f1f0b8b8d1c8760725970f97546
  Cloning https://github.com/bowang-lab/scGPT.git (to revision cebd6fae655b9c585a4807daa3ac31bb764f06b4) to /tmp/pip-req-build-6mz7d6r0
  Running command git clone --filter=blob:none --quiet https://github.com/bowang-lab/scGPT.git /tmp/pip-req-build-6mz7d6r0
  Running command git rev-parse -q --verify 'sha^cebd6fae655b9c585a4807daa3ac31bb764f06b4'
  Running command git fetch -q https://github.com/bowang-lab/scGPT.git cebd6fae655b9c585a4807daa3ac31bb764f06b4
  Resolved https://github.com/bowang-lab/scGPT.git to commit cebd6fae655b9c585a4807daa3ac31bb764f06b4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
peft: 0.19.1
scGPT commit: cebd6fa | checkpoint: /content/drive/MyDrive/ad-glia-fm-prep/scgpt_whole_hu

> **Interpretation — environment installed clean, checkpoint reused, real GPU confirmed, the full `MEASURE_FLOOR` path taken (1a).**
>
> `MEASURE_FLOOR=True` this run, so the branch that installs scGPT fired: `--no-deps` at the pinned commit `cebd6fa` (the same commit colab_16's CPT checkpoint was produced under), then `requirements_scgpt.txt`, then a `pip uninstall torchao` step -- the same pre-emptive step colab_16 also carries, since peft 0.19.1's `is_torchao_available()` raises on Colab's stock torchao 0.10.0 instead of returning `False`, which would otherwise kill the adapter reload in 5b. As in colab_16, this step was a no-op on this run too (`WARNING: Skipping torchao as it is not installed`) -- Colab's runtime shipped without it again, so its actual removal has still never been exercised by either notebook. `peft` is asserted `==0.19.1`, not just installed, because 5b's `PeftModel.from_pretrained` + `merge_and_unload()` depend on this exact version's `nn.MultiheadAttention` merge behaviour.
>
> The pretrained whole-human checkpoint was already present on Drive from an earlier session -- `_have_ckpt` returned `True` for all three files, so the `gdown` download branch never ran; the checkpoint was reused, not re-downloaded. GPU is confirmed as a real NVIDIA A100-SXM4-80GB by the same hard `torch.cuda.is_available()` assert this project's foundation-model notebooks use to refuse a CPU-only run outright. `SMOKE=False` and `MEASURE_FLOOR=True` -- every path from here on is the real run plus the full six-pass floor measurement, not a rehearsal.

### 1b — pip freeze + env JSON (records the exact eval-run stack)

`scgpt_commit` is written as `null` on a tier-1 run and rewritten at 4a once the audit trail says
which commit the embeddings came from.

In [2]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_17"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze{SUFFIX}.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        m = __import__(mod)
    except Exception as e:
        print(f"  [_ver] import '{mod}' not available: {type(e).__name__}: {e}")
        return None
    try:
        return m.__version__
    except AttributeError:
        import importlib.metadata as ilm
        try:
            return ilm.version(mod)
        except Exception:
            return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "smoke":          bool(SMOKE),
    "measure_floor":  bool(MEASURE_FLOOR),
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "scgpt_commit":     SCGPT_COMMIT,
    "scgpt_version":    _ver("scgpt") if MEASURE_FLOOR else None,
    "peft_version":     _ver("peft") if MEASURE_FLOOR else None,
    "scanpy_version":   _ver("scanpy"),
    "anndata_version":  _ver("anndata"),
    "sklearn_version":  _ver("sklearn"),
    "torch_version":    _ver("torch") if MEASURE_FLOOR else None,
    "numpy_version":    _ver("numpy"),
    "model_checkpoint": os.path.basename(MODEL_DIR),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env{SUFFIX}.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

{
  "notebook_id": "colab_17",
  "date": "2026-07-27",
  "smoke": false,
  "measure_floor": true,
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "os_release": "6.6.122+",
  "gpu": "GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-f7b34233-7eed-40f6-7384-dc4832631580)",
  "cuda": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "git_commit": "899bfd338ed51f1f0b8b8d1c8760725970f97546",
  "scgpt_commit": "cebd6fae655b9c585a4807daa3ac31bb764f06b4",
  "scgpt_version": "0.2.5",
  "peft_version": "0.19.1",
  "scanpy_version": "1.12.3",
  "anndata_version": "0.13.2",
  "sklearn_version": "1.6.1",
  "torch_version": "2.11.0+cu128",
  "numpy_version": "2.4.6",
  "model_checkpoint": "scgpt_whole_human"
}


/tmp/ipykernel_4532/2153338505.py:25: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  return m.__version__
/tmp/ipykernel_4532/2153338505.py:25: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  return m.__version__


> **Interpretation — environment snapshot recorded, matches colab_16 exactly on every shared component (1b).**
>
> Python 3.12.13, torch 2.11.0+cu128, peft 0.19.1, scgpt 0.2.5, scanpy 1.12.3, anndata 0.13.2, scikit-learn 1.6.1, numpy 2.4.6, repo commit `899bfd3` (the commit that carries this notebook's pre-run fixes), scGPT commit `cebd6fa` -- an identical FM/library stack to colab_16's CPT run, so nothing about this notebook's own environment can account for any difference between what it measures and what colab_16 recorded. The two `FutureWarning`s on `scanpy`/`anndata` `__version__` access are the same cosmetic deprecation colab_16 saw -- the read still succeeds, the warning just flags the attribute-access style as outdated.

## 2 — Substrate and schema

### 2a — Rebuild the glia substrate (deterministic; the same `cell_index` as every saved embedding)

Rebuilt from the colab_07 / colab_08 labelled subsets in the same order as every prior FM notebook,
so `cell_index = arange(n_obs)` lines up cell-for-cell with the embeddings loaded in 6a. `region`
rides along on the subset files (they copy the parent and never drop obs), so eval #2's confound
audit needs no barcode join.

The count matrix is only needed when `MEASURE_FLOOR=True` — the input transform is applied inside
that branch, identical to the one colab_10 and colab_16 used, since any deviation would register
downstream as drift that CPT did not cause.

In [3]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:28s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape)
print("astrocyte subset:", astro.shape)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "region", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
glia.var["gene_name"] = glia.var_names          # explicit symbol column for scGPT's gene lookup
print("\ncombined glia:", glia.shape)

REF_N_CELLS = 142588
REF_N_GENES = 26514
REF_LINEAGE = {"astrocyte": 87783, "microglia": 54805}
assert glia.n_obs == REF_N_CELLS, f"substrate cell count {glia.n_obs} != reference {REF_N_CELLS}"
assert glia.n_vars == REF_N_GENES, f"gene panel {glia.n_vars} != reference {REF_N_GENES}"
lin_counts = glia.obs["lineage"].value_counts().to_dict()
assert lin_counts == REF_LINEAGE, f"lineage counts {lin_counts} != reference {REF_LINEAGE}"
print(f"substrate OK: {glia.n_obs} cells x {glia.n_vars} genes | lineage {lin_counts}")

# raw-counts guard -- the transform below must START from raw counts, as it did in colab_10/16.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}) -- FM input must be raw"
print("raw-counts int-frac:", round(frac_int, 3))

if MEASURE_FLOOR:
    # Identical to the zero-shot baseline's input transform (colab_10 3a) and colab_16's (3a).
    sc.pp.normalize_total(glia, target_sum=1e4)
    sc.pp.log1p(glia)
    print("applied normalize_total(1e4) + log1p (needed for the embedding passes in 5b)")
else:
    print("MEASURE_FLOOR=False -- .X is not used by the eval path; left untransformed, freed in 6a")
_ram("combined glia")

microglia subset: (54805, 26514)
astrocyte subset: (87783, 26514)

combined glia: (142588, 26514)
substrate OK: 142588 cells x 26514 genes | lineage {'astrocyte': 87783, 'microglia': 54805}
raw-counts int-frac: 1.0
applied normalize_total(1e4) + log1p (needed for the embedding passes in 5b)
[RAM] combined glia               :   7.2 / 179.4 GB (5%)


> **Interpretation — substrate rebuilt exactly, raw counts confirmed, same transform as the embeddings being read (2a).**
>
> Reloading the labelled microglia (54,805 cells) and astrocyte (87,783 cells) subsets and concatenating them reproduces the frozen 142,588-cell / 26,514-gene glia substrate every FM notebook in this project has used since colab_09. The raw-counts guard sampled 2,000 cells at random and found an exact-integer fraction of 1.0, confirming the upstream files still carry untouched raw counts. Because `MEASURE_FLOOR=True`, `normalize_total(1e4)` + `log1p` are then applied -- the identical two-step transform colab_10's zero-shot baseline and colab_16's CPT run both used. That match is what licenses reading this run's fresh floor-measurement passes (5b) against the stored zero-shot/CPT matrices without a preprocessing mismatch as a confound: any difference the later cells find has to come from the model, not from how the input was scaled. RAM sits at 7.2 of 179.4 GB (5%) -- comfortable headroom for everything still to come.

### 2b — Fail loud on the obs schema the evals depend on

Every column below is load-bearing for a specific audit or eval slice, so a missing or null one is a
hard stop rather than a silently dropped subgroup.

In [4]:
REQUIRED_OBS = ["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "region", "donor_id"]
missing_cols = [c for c in REQUIRED_OBS if c not in glia.obs.columns]
assert not missing_cols, (
    f"substrate missing required obs columns: {missing_cols} "
    "(region is inherited from the colab_07/08 subsets -- check they carry it)")

for col in REQUIRED_OBS:
    n_null = int(pd.isna(glia.obs[col]).sum())
    assert n_null == 0, f"{col} has {n_null} null values -- the eval audits require it complete"

assert set(glia.obs["lineage"]) == {"microglia", "astrocyte"}, "unexpected lineage values"
assert set(glia.obs["substate"]) <= {"homeostatic", "activated", "resting", "reactive", "intermediate"}, \
    f"unexpected substate values: {set(glia.obs['substate'])}"
assert set(glia.obs["apoe_carrier"]) <= {"carrier", "noncarrier", "e2"}, \
    f"unexpected apoe_carrier values: {set(glia.obs['apoe_carrier'])}"
assert set(glia.obs["study_id"]) == {"SEA-AD", "Li2025", "Haney2024"}, \
    f"unexpected studies: {set(glia.obs['study_id'])}"

print("obs schema OK")
print("\nlineage x substate:")
print(pd.crosstab(glia.obs["lineage"], glia.obs["substate"]))
print("\napoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())
print("donors:", glia.obs["donor_id"].nunique())
print("\nregion x study:")
print(pd.crosstab(glia.obs["region"], glia.obs["study_id"]))
print("\nNOTE: Haney2024's region column carries the literal string 'unknown' for all of its cells")
print("      (non-null, so the check above legitimately passes) -- eval #2's region confound table")
print("      in 8a therefore buckets all of Haney into one uninformative group.")

obs schema OK

lineage x substate:
substate   activated  homeostatic  intermediate  reactive  resting
lineage                                                           
astrocyte          0            0         11171     28465    48147
microglia      12109        25845         16851         0        0

apoe_carrier: {'noncarrier': 70169, 'carrier': 49831, 'e2': 22588}
donors: 145

region x study:
study_id         Li2025  SEA-AD  Haney2024
region                                    
MTG                   0   75634          0
temporal cortex   46870       0          0
unknown               0       0      20084

NOTE: Haney2024's region column carries the literal string 'unknown' for all of its cells
      (non-null, so the check above legitimately passes) -- eval #2's region confound table
      in 8a therefore buckets all of Haney into one uninformative group.


> **Interpretation — obs schema passes, composition matches every prior notebook, Haney's region caveat still live (2b).**
>
> The lineage x substate crosstab reproduces the same five-bucket split every Geneformer and scGPT notebook in this project has reported since colab_07/08: astrocyte 48,147 resting / 28,465 reactive / 11,171 intermediate; microglia 25,845 homeostatic / 12,109 activated / 16,851 intermediate. `apoe_carrier` breaks down as 70,169 noncarrier / 49,831 carrier / 22,588 e2 -- e2 is not excluded here, only at the eval-specific cells (8a/8b) that actually score the APOE axis. All 145 donors are present.
>
> The region x study table carries forward a known, non-ideal fact rather than papering over it: Haney2024's `region` column reads the literal string `"unknown"` for all 20,084 of its cells -- non-null, so the schema assert legitimately passes, but not a real anatomical label. This cell's own print statement flags it explicitly so it can't be missed downstream: 8a's confound table will bucket all of Haney2024 into one uninformative row rather than a real region breakdown.

## 3 — Held-out split verification

### 3a — Rebuild the frozen donor split and hard-stop on any mismatch

The donor-level split is frozen for the whole project (`outputs/donor_split.json`). A mismatch here
is a hard stop, never a new split: these evals are only comparable to colab_12 / colab_15 / colab_16
if they score the same held-out donors. Donor *identity* is checked, not just donor counts — a
count-only check passes on a reshuffled split.

In [5]:
SPLIT_PATH = os.path.join(REPO_PATH, "outputs", "donor_split.json")
assert os.path.exists(SPLIT_PATH), f"missing frozen split {SPLIT_PATH}"
with open(SPLIT_PATH) as f:
    split_artifact = json.load(f)

REF_SEED       = 32
REF_MARGIN     = 10
REF_N_DONORS   = {"train": 101, "val": 22, "test": 22}
REF_N_CELLS_SP = {"train": 94963, "val": 23824, "test": 23801}

assert int(split_artifact["seed"]) == REF_SEED, f"split seed {split_artifact['seed']} != {REF_SEED}"
assert int(split_artifact["test_worst_case_margin"]) == REF_MARGIN, "split test margin != reference"
assert {k: int(v) for k, v in split_artifact["n_donors"].items()} == REF_N_DONORS, \
    "split donor counts != reference"

split_map = split_artifact["donor_split"]
glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map)
assert not glia.obs["split"].isna().any(), "some substrate donors are absent from the frozen split"
glia.obs["split"] = glia.obs["split"].astype("category")

cell_counts = glia.obs["split"].value_counts().to_dict()
assert {k: int(v) for k, v in cell_counts.items()} == REF_N_CELLS_SP, (
    f"cells per split {cell_counts} != reference {REF_N_CELLS_SP}")

n_mismatch = 0
for part in ("train", "val", "test"):
    frozen = {d for d, s in split_map.items() if s == part}
    here   = set(glia.obs.loc[glia.obs["split"] == part, "donor_id"].astype(str))
    if frozen != here:
        n_mismatch += len(frozen ^ here)
        print(f"  MISMATCH {part}: {sorted(frozen ^ here)[:5]}")
assert n_mismatch == 0, f"{n_mismatch} donors differ between the frozen split and this substrate"
print(f"split verified: seed {REF_SEED}, margin {REF_MARGIN}, donors {REF_N_DONORS}, "
      f"cells {cell_counts}, donor identities match for all {len(split_map)} donors")

test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True)
print("test-set study fractions:", test_by_study.round(3).to_dict())
assert test_by_study.max() <= 0.60 + 1e-9, (
    f"a single study is {test_by_study.max():.1%} of the held-out set -- above the 60% composition "
    "bar in docs/setup_audits.md; any result would be a composition artifact")

# --- smoke subsample (last, and only here) ------------------------------------------------
if SMOKE:
    rng = np.random.default_rng(0)
    keep, starved = [], []
    grp = glia.obs.groupby(["split", "lineage", "substate"], observed=True).indices
    for k, idx in grp.items():
        take = min(SMOKE_CAP, len(idx))
        if take < SMOKE_CAP:
            starved.append((k, len(idx)))
        keep.append(rng.choice(idx, size=take, replace=False))
    keep = np.sort(np.concatenate(keep))
    glia = glia[keep].copy()
    print(f"\nSMOKE subsample: {glia.n_obs} cells from {len(grp)} groups (cap {SMOKE_CAP}/group)")
    if starved:
        print(f"  {len(starved)} group(s) below cap:", starved[:5])
    print("  split:", glia.obs["split"].value_counts().to_dict())
    _ram("after smoke subsample")

split verified: seed 32, margin 10, donors {'train': 101, 'val': 22, 'test': 22}, cells {'train': 94963, 'val': 23824, 'test': 23801}, donor identities match for all 145 donors
test-set study fractions: {'SEA-AD': 0.567, 'Li2025': 0.3, 'Haney2024': 0.133}


> **Interpretation — frozen donor split reproduced exactly, held-out composition within the 60% ceiling (3a).**
>
> Seed 32, margin 10, donor counts {101, 22, 22}, cell counts {94,963, 23,824, 23,801} -- all match the hardcoded reference exactly, and the per-donor identity loop found zero mismatches across all 145 donors. As in colab_16, that identity loop is weaker than it looks: both sides (`here` and `frozen`) are derived from the same loaded `donor_split.json`, so it can only catch a donor the split map assigns to a partition that ends up contributing zero cells to this substrate, not a reshuffled or substituted split file. What actually pins this run to the one partition every other FM notebook has used is the hardcoded seed/margin/count asserts above, together with `donor_split.json` never having been regenerated since it was first committed.
>
> Test-set study composition -- SEA-AD 56.7% / Li2025 30.0% / Haney2024 13.3% -- sits under the 60% single-study ceiling from `docs/setup_audits.md`, the same uneven-by-construction mix every held-out evaluation in this project has used. `SMOKE=False`, so the subsample block at the bottom of this cell never executed -- the full 142,588-cell substrate carries into every cell from here on.

## 4 — Embedding inventory

### 4a — Inventory the embeddings and the detector #1 numbers from the recorded audit trail

Paths, the adapter location and every detector #1 quantity are **read from `audit_report.json`**,
not reconstructed from filename conventions — the audit trail is the authority on what colab_10 and
colab_16 actually produced.

Both embedding sets must come from one scGPT commit: a different commit means a different vocabulary
or input encoding, which would make the Δ a mix of model change and tokenizer change.

The repeat-pass files are inventoried here too. Computing them is gated by `MEASURE_FLOOR`; *using*
them is gated only by whether they exist, so a GPU run's passes stay available to every later
CPU-only run.

In [7]:
AUDIT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_PATH) as f:
    audit = json.load(f)

zs  = audit["scgpt_zeroshot"]
cpt = audit["scgpt_cpt_aggregated"]
assert zs["status"] == "computed",  "colab_10's scGPT zero-shot run is not recorded as computed"
assert cpt["status"] == "computed", "colab_16's scGPT CPT run is not recorded as computed"
assert cpt["fm"] == "scgpt" and cpt["regime"] == "aggregated", "unexpected CPT record"
for rec, name in ((zs, "zero-shot"), (cpt, "CPT")):
    assert rec["n_cells"] == REF_N_CELLS, \
        f"the {name} embedding covers {rec['n_cells']} cells, not the {REF_N_CELLS} substrate"

_commits = {zs["scgpt_commit"], cpt["scgpt_commit"]}
assert len(_commits) == 1, f"the two embeddings span multiple scGPT commits: {_commits}"
EMB_COMMIT = _commits.pop()
if SCGPT_COMMIT is None:
    # tier-1 run: scGPT is not installed, so adopt the commit the embeddings were recorded under
    # and rewrite the env JSON, which 1b wrote as null before this was knowable.
    SCGPT_COMMIT = EMB_COMMIT
    env_snapshot["scgpt_commit"] = EMB_COMMIT
    env_snapshot["scgpt_commit_source"] = "audit_report.json (scGPT not installed this run)"
    with open(ENV_JSON_PATH, "w") as f:
        json.dump(env_snapshot, f, indent=2)
else:
    assert SCGPT_COMMIT == EMB_COMMIT, (
        f"installed scGPT {SCGPT_COMMIT[:7]} != the commit that produced the existing embeddings "
        f"{EMB_COMMIT[:7]} -- new passes would not be comparable to the stored matrices")
print("embedding scGPT commit:", EMB_COMMIT[:7])

PATHS = {"zeroshot": os.path.join(DRIVE_ROOT, zs["embedding_file"]),
         "cpt":      os.path.join(DRIVE_ROOT, cpt["embedding_file"])}
for k, p in PATHS.items():
    assert os.path.exists(p), f"missing {k} embedding {p} (colab_10 / colab_16 output must be on Drive)"
    print(f"  {k:9s} OK  {os.path.relpath(p, DRIVE_ROOT)}")

ADAPTER_DIR = os.path.join(DRIVE_ROOT, cpt["adapter_file"])
EMB_DIM     = cpt["emb_dim"]
MAX_LENGTH  = cpt["max_length"]
LORA_TARGETS_RECORDED = cpt["lora"]["targets"]

# --- detector #1, as colab_16 recorded it: the magnitudes these evals are read against ---
D1            = cpt["detector_1"]
DRIFT_ALL_REC = D1["drift_all"]
DRIFT_TEST_REC = D1["drift_test"]
FLOOR_D1      = D1["noise_floor_measured"]
SUBSTATE_REF  = cpt["substate_reference_scgpt_space"]
PCT_OF_REF    = cpt["drift_pct_of_substate_reference"]
assert not D1["inert"], "colab_16 recorded the CPT checkpoint as inert -- there is nothing to evaluate"

print(f"\ndetector #1 (colab_16): drift_all {DRIFT_ALL_REC:.5f} | drift_test {DRIFT_TEST_REC:.5f}")
print(f"  measured noise floor {FLOOR_D1:.5f} -> drift is {D1['drift_over_floor']:.2f}x the floor")
print(f"  within-donor substate reference (scGPT space): "
      f"micro {SUBSTATE_REF['microglia']['within_donor']:.5f} "
      f"({SUBSTATE_REF['microglia']['n_donors']} donors) | "
      f"astro {SUBSTATE_REF['astrocyte']['within_donor']:.5f} "
      f"({SUBSTATE_REF['astrocyte']['n_donors']} donors)")
print(f"  drift as % of that reference: micro {PCT_OF_REF['microglia']}% | astro {PCT_OF_REF['astrocyte']}%")
print("  (these are cosine-distance quantities, 1-cos(theta); a ratio of them is NOT a ratio of angles)")
print(f"  emb_dim {EMB_DIM} | max_length {MAX_LENGTH} | LoRA targets {LORA_TARGETS_RECORDED}")

# --- repeat-pass inventory -----------------------------------------------------------------
FLOOR_DIR = os.path.join(DRIVE_ROOT, "scgpt", "floor")

def _pass_path(kind, i):
    # commit prefix in the filename so a pass made under a different scGPT commit can never be
    # silently reloaded as if it were comparable (the colab_13 cache-key lesson).
    return os.path.join(FLOOR_DIR, f"glia_scgpt_{kind}_p{i}_c{SCGPT_COMMIT[:7]}{SUFFIX}.h5ad")

BASE_PASS_PATHS = [_pass_path("base", i)      for i in range(1, N_BASE_PASSES + 1)]
CPT_PASS_PATHS  = [_pass_path("cptmerged", i) for i in range(1, N_CPT_PASSES + 1)]
FLOOR_AVAILABLE = all(os.path.exists(p) for p in BASE_PASS_PATHS + CPT_PASS_PATHS)

print(f"\nrepeat passes for the measured noise floor (floor dir {os.path.relpath(FLOOR_DIR, DRIVE_ROOT)}):")
for p in BASE_PASS_PATHS + CPT_PASS_PATHS:
    state = "OK      " if os.path.exists(p) else ("to build" if MEASURE_FLOOR else "absent  ")
    print(f"  {state} {os.path.basename(p)}")
_n_present = sum(os.path.exists(p) for p in BASE_PASS_PATHS + CPT_PASS_PATHS)
print(f"FLOOR_AVAILABLE={FLOOR_AVAILABLE} ({_n_present} of {N_BASE_PASSES + N_CPT_PASSES} passes present)")
if MEASURE_FLOOR:
    assert os.path.isdir(ADAPTER_DIR), (
        f"MEASURE_FLOOR=True needs colab_16's LoRA adapter at {ADAPTER_DIR}; not found on Drive. "
        "colab_16 verified its embedding FILE on reload, never the adapter directory.")
    print("adapter dir OK:", os.path.relpath(ADAPTER_DIR, DRIVE_ROOT))
elif not FLOOR_AVAILABLE:
    print("\nNOTE: no measured noise floor this run. Every Δ below carries its pre-registered")
    print("      contract band only, with no measured null to say whether a small Δ is")
    print("      distinguishable from scGPT's re-embedding stochasticity.")

embedding scGPT commit: cebd6fa
  zeroshot  OK  scgpt/glia_scgpt_zeroshot.h5ad
  cpt       OK  scgpt/glia_scgpt_cpt_aggregated_seed0.h5ad

detector #1 (colab_16): drift_all 0.08090 | drift_test 0.08115
  measured noise floor 0.00365 -> drift is 22.25x the floor
  within-donor substate reference (scGPT space): micro 0.03598 (36 donors) | astro 0.05256 (78 donors)
  drift as % of that reference: micro 234.5% | astro 149.3%
  (these are cosine-distance quantities, 1-cos(theta); a ratio of them is NOT a ratio of angles)
  emb_dim 512 | max_length 1200 | LoRA targets transformer_encoder\.layers\.\d+\.(self_attn|linear1|linear2)$

repeat passes for the measured noise floor (floor dir scgpt/floor):
  to build glia_scgpt_base_p1_ccebd6fa.h5ad
  to build glia_scgpt_base_p2_ccebd6fa.h5ad
  to build glia_scgpt_base_p3_ccebd6fa.h5ad
  to build glia_scgpt_cptmerged_p1_ccebd6fa.h5ad
  to build glia_scgpt_cptmerged_p2_ccebd6fa.h5ad
  to build glia_scgpt_cptmerged_p3_ccebd6fa.h5ad
FLOOR_AVAILABLE=Fals

> **Interpretation — colab_16's recorded numbers loaded and cross-checked; the adapter directory, not just the embedding file, confirmed present (4a).**
>
> This cell doesn't recompute anything yet -- it reads colab_16's own `audit_report.json` entries and checks several things about them before trusting either as this run's starting point: both records are asserted `status=="computed"`, both cover the full `n_cells==142588` substrate, both share a single scGPT commit (`cebd6fa`, matching what this run just installed), and the CPT record is asserted `not inert`. Detector #1 as colab_16 recorded it: `drift_all` 0.08090, `drift_test` 0.08115, measured noise floor 0.00365, drift 22.25x the floor; within-donor substate reference in scGPT's own space, microglia 0.03598 (36 donors) / astrocyte 0.05256 (78 donors); drift as a percentage of that reference, 234.5% micro / 149.3% astro. The printed reminder that these are `1-cos(theta)` quantities, not angles, carries forward colab_16's own correction of what a ratio of them means -- a ratio of cosine-distances is not a ratio of rotation angles.
>
> A genuinely new check this cell adds, beyond restating colab_16: it asserts the LoRA **adapter directory** exists on Drive (`ADAPTER_DIR`), a prerequisite colab_16's own reload never actually verified -- that notebook only ever confirmed its embedding *file* on reload, not the adapter directory this run's floor-measurement pass depends on. The floor-pass inventory printed at the end shows 0 of 6 repeat-embedding files present yet -- building them is exactly what 5a/5b do next.

## 5 — Measured noise floor for the eval metrics (`MEASURE_FLOOR` only)

### 5a — Vocabulary, input geometry, dataset and model builders

Everything in this section is skipped when `MEASURE_FLOOR=False`. The vocabulary intersection, the
in-vocab gene subset and the per-cell encoding are rebuilt exactly as colab_16 built them, because a
pass that encoded cells differently would measure that difference rather than the sampling noise it
is supposed to measure. Coverage is cross-checked against the fraction colab_10 recorded.

In [8]:
if not MEASURE_FLOOR:
    print("skipped -- MEASURE_FLOOR=False (5b is skipped too; 6a uses any passes already on Drive)")
else:
    import re
    import torch
    from scgpt.tokenizer import GeneVocab
    from scgpt.model import TransformerModel
    from scgpt.data_collator import DataCollator
    from torch.utils.data import DataLoader, SequentialSampler

    DEVICE = torch.device("cuda")

    # PyTorch's attention modules carry an inference fast path that reads in_proj_weight as a raw
    # tensor instead of calling the module's forward. Disabled explicitly so nothing here depends
    # on autocast implicitly skipping it, exactly as colab_16 did.
    torch.backends.mha.set_fastpath_enabled(False)
    print("attention fast path enabled:", torch.backends.mha.get_fastpath_enabled())

    # ---------------------------------------------------------------- vocabulary
    VOCAB_FILE  = os.path.join(MODEL_DIR, "vocab.json")
    CONFIG_FILE = os.path.join(MODEL_DIR, "args.json")
    with open(CONFIG_FILE) as f:
        model_configs = json.load(f)

    PAD_TOKEN = "<pad>"
    vocab = GeneVocab.from_file(VOCAB_FILE)
    for s in [PAD_TOKEN, "<cls>", "<eoc>"]:
        if s not in vocab:
            vocab.append_token(s)
    print("scGPT vocabulary size:", len(vocab))

    glia.var["id_in_vocab"] = [vocab[g] if g in vocab else -1 for g in glia.var["gene_name"]]
    n_total = glia.n_vars
    n_vocab = int((glia.var["id_in_vocab"] >= 0).sum())
    frac_here = round(n_vocab / n_total, 4)
    assert frac_here == zs["vocab_audit"]["frac_in_vocab"], (
        f"in-vocab fraction {frac_here} != the zero-shot record "
        f"{zs['vocab_audit']['frac_in_vocab']} -- the gene panel or the vocabulary changed, so "
        "these passes would not be comparable to the stored matrices")
    assert "APOE" in vocab, "APOE absent from the scGPT vocabulary -- eval #2 cannot be run"
    print(f"gene panel {n_total} | in vocab {n_vocab} ({frac_here:.1%}) -- agrees with colab_10")

    glia_v = glia[:, glia.var["id_in_vocab"] >= 0].copy()
    vocab.set_default_index(vocab[PAD_TOKEN])
    GENE_IDS  = np.array(vocab(glia_v.var["gene_name"].tolist()), dtype=int)
    assert len(GENE_IDS) == glia_v.n_vars, "gene-id array is not aligned to the in-vocab panel"
    CLS_ID    = vocab["<cls>"]
    PAD_ID    = vocab[PAD_TOKEN]
    PAD_VALUE = model_configs["pad_value"]
    print(f"in-vocab matrix: {glia_v.shape} | <cls> {CLS_ID} | <pad> {PAD_ID} | pad_value {PAD_VALUE}")

    # ---------------------------------------------------------------- input geometry
    Xv = glia_v.X
    assert sp.issparse(Xv), "expected a sparse in-vocab matrix"
    Xv = Xv.tocsr(); Xv.eliminate_zeros()
    nnz_per_cell = np.diff(Xv.indptr)
    seq_len = nnz_per_cell + 1                      # +1 for the prepended <cls> token
    over = int((seq_len > MAX_LENGTH).sum())
    assert int((nnz_per_cell == 0).sum()) == 0, "a cell has no detected in-vocab gene"
    print(f"\ncells above the {MAX_LENGTH}-token context: {over} ({over/len(seq_len):.1%})")
    print("  -> these are RANDOMLY SUBSAMPLED on every pass, which is exactly the stochasticity")
    print("     the repeat passes below are here to quantify at the level of the eval metrics.")
    if not SMOKE:
        rec_over = cpt["frac_cells_over_context"]
        assert abs(over/len(seq_len) - rec_over) < 1e-3, (
            f"fraction over context {over/len(seq_len):.4f} != colab_16's recorded {rec_over}")

    INT32MAX  = 2**31 - 1
    N_HEADS   = model_configs["nheads"]
    EMB_BATCH = 64
    while EMB_BATCH > 1 and EMB_BATCH * N_HEADS * MAX_LENGTH**2 >= INT32MAX:
        EMB_BATCH //= 2
    _elems = EMB_BATCH * N_HEADS * MAX_LENGTH**2
    assert _elems < INT32MAX, "attention score tensor exceeds the int32 limit even at batch 1"
    print(f"batch geometry OK: {EMB_BATCH} x {N_HEADS} heads x {MAX_LENGTH}^2 = {_elems:,} < {INT32MAX:,}")

    # ---------------------------------------------------------------- dataset + collator
    NUM_WORKERS = 2

    class GliaCellDataset(torch.utils.data.Dataset):
        """One cell -> (gene ids, expression values) with <cls> prepended, densified per row."""
        def __init__(self, X_csr, gene_ids, cls_id, pad_value):
            self.X, self.gene_ids, self.cls_id, self.pad_value = X_csr, gene_ids, cls_id, pad_value
        def __len__(self):
            return self.X.shape[0]
        def __getitem__(self, idx):
            row = self.X[idx].toarray().ravel()
            nz = np.nonzero(row)[0]
            genes  = np.insert(self.gene_ids[nz], 0, self.cls_id)
            values = np.insert(row[nz], 0, self.pad_value)
            return {"id": idx,
                    "genes": torch.from_numpy(genes).long(),
                    "expressions": torch.from_numpy(values).float()}

    full_ds = GliaCellDataset(Xv, GENE_IDS, CLS_ID, PAD_VALUE)

    def make_collator():
        # sampling=True is what draws the random gene subsample for over-context cells, and the
        # collator runs per BATCH -- so a fresh collator per pass genuinely redraws it. do_mlm is
        # False: this is an embedding path, not a loss path.
        return DataCollator(
            do_padding=True, pad_token_id=PAD_ID, pad_value=PAD_VALUE,
            do_mlm=False, do_binning=True, mask_value=-1, max_length=MAX_LENGTH,
            sampling=True, keep_first_n_tokens=1,
        )

    # ---------------------------------------------------------------- model builders
    def build_scgpt(model_configs, vocab):
        """Construct the model with the same arguments the library's embedding path uses."""
        return TransformerModel(
            ntoken=len(vocab), d_model=model_configs["embsize"], nhead=model_configs["nheads"],
            d_hid=model_configs["d_hid"], nlayers=model_configs["nlayers"],
            nlayers_cls=model_configs["n_layers_cls"], n_cls=1, vocab=vocab,
            dropout=model_configs["dropout"], pad_token=model_configs["pad_token"],
            pad_value=model_configs["pad_value"], do_mvc=True, do_dab=False,
            use_batch_labels=False, domain_spec_batchnorm=False, explicit_zero_prob=False,
            use_fast_transformer=False,     # no flash-attn -> PyTorch attention, as in the baseline
            fast_transformer_backend="flash", pre_norm=False,
        )

    # The released checkpoint stores fused attention weights under the flash-attention naming.
    RENAME_RULES = {
        r"self_attn\._impl\.Wqkv\.": "self_attn.in_proj_",
        r"self_attn\.Wqkv\.": "self_attn.in_proj_",
        r"self_attn\._impl\.out_proj\.": "self_attn.out_proj.",
    }
    REQUIRED_PREFIXES = ("encoder.", "value_encoder.", "transformer_encoder.", "decoder.")

    def load_checkpoint_verbose(model, ckpt_path):
        """Load the checkpoint and REPORT what was dropped -- the loader itself drops silently."""
        raw = torch.load(ckpt_path, map_location="cpu")
        renamed = {}
        for k, v in raw.items():
            kk = k
            for pat, rep in RENAME_RULES.items():
                kk = re.sub(pat, rep, kk)
            renamed[kk] = v
        model_dict = model.state_dict()
        kept    = {k: v for k, v in renamed.items() if k in model_dict and v.shape == model_dict[k].shape}
        dropped = sorted(set(renamed) - set(kept))
        model_dict.update(kept)
        model.load_state_dict(model_dict)
        return kept, dropped

    def assert_required_loaded(model, kept, tag):
        """Every parameter on the input->readout path must have come from the checkpoint."""
        missing = [n for n, _ in model.named_parameters()
                   if n.startswith(REQUIRED_PREFIXES) and n not in kept]
        assert not missing, (
            f"[{tag}] {len(missing)} parameter(s) on the input->readout path were NOT loaded from "
            f"the checkpoint, e.g. {missing[:8]}")
        print(f"[{tag}] all required parameter groups loaded from the checkpoint")

    @torch.no_grad()
    def embed_cells(model, dataset, desc=""):
        """<cls>-position cell embeddings, L2-normalized -- the same readout as every stored matrix."""
        from tqdm.auto import tqdm
        loader = DataLoader(dataset, batch_size=EMB_BATCH, sampler=SequentialSampler(dataset),
                            collate_fn=make_collator(), drop_last=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        model.eval()
        out = np.zeros((len(dataset), model_configs["embsize"]), dtype=np.float32)
        count = 0
        for batch in tqdm(loader, desc=desc or "embedding"):
            gene = batch["gene"].to(DEVICE, non_blocking=True)
            expr = batch["expr"].to(DEVICE, non_blocking=True)
            pad_mask = gene.eq(PAD_ID)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                h = model._encode(gene, expr, src_key_padding_mask=pad_mask, batch_labels=None)
            e = h[:, 0, :].float().cpu().numpy()          # <cls> position
            out[count:count + len(e)] = e
            count += len(e)
        assert count == len(dataset), f"embedded {count} of {len(dataset)} cells"
        return out / (np.linalg.norm(out, axis=1, keepdims=True) + 1e-12)

    print(f"\nbuilders ready | dataset {len(full_ds)} cells | emb batch {EMB_BATCH}")
    _ram("floor tier ready")

attention fast path enabled: False
scGPT vocabulary size: 60697
gene panel 26514 | in vocab 19977 (75.3%) -- agrees with colab_10
in-vocab matrix: (142588, 19977) | <cls> 60695 | <pad> 60694 | pad_value -2

cells above the 1200-token context: 123112 (86.3%)
  -> these are RANDOMLY SUBSAMPLED on every pass, which is exactly the stochasticity
     the repeat passes below are here to quantify at the level of the eval metrics.
batch geometry OK: 64 x 8 heads x 1200^2 = 737,280,000 < 2,147,483,647

builders ready | dataset 142588 cells | emb batch 64
[RAM] floor tier ready            :  10.6 / 179.4 GB (7%)


> **Interpretation — vocabulary and batch geometry match the zero-shot baseline exactly; no int32-overflow exposure at this context length (5a).**
>
> The attention fast path is explicitly disabled (as in colab_16), so the module's real forward runs instead of a raw-tensor inference shortcut. Vocabulary size 60,697; the in-vocab fraction is re-derived here (75.3%, 19,977 of 26,514 genes) and hard-asserted equal to colab_10's recorded fraction -- an exact-match check, not a visual comparison. 86.3% of cells (123,112 of 142,588) exceed the 1,200-token context and get randomly subsampled on every pass -- the single fact this whole notebook exists to quantify, since it is what makes re-embedding the *same* cell twice non-deterministic and gives detector #1's noise floor a real, nonzero value to measure rather than assume (unlike Geneformer's exactly-0.0000 floor).
>
> The batch-geometry check confirms this run was never at risk of the earlier int32-overflow crash class: at `EMB_BATCH=64`, 8 heads, length 1,200, the attention score tensor holds 737,280,000 elements, comfortably under the ~2.147-billion signed-int32 ceiling -- roughly 6.6x smaller than the tensor that actually crashed a Geneformer notebook at the length (2,516 tokens) that triggered it, so no adaptive batch-size reduction was needed here at all.

### 5b — Repeat embedding passes: frozen base x N, merged adapter x N

Each pass is written to Drive and dropped from memory, so 6a loads every matrix through one uniform
path and the numbers below are reproducible from disk without a GPU. Existing passes are skipped, so
re-running this cell is free.

The adapter is reloaded with `PeftModel.from_pretrained` and merged. peft actually **raises** if a
target pattern matches nothing at all anywhere in the model — it doesn't fail silently there. But
that isn't the only way a reload could go wrong, so the adapter is still explicitly asserted to be
attached to the encoder layers (and asserted *not* to have leaked into the `ContinuousValueEncoder`,
whose `linear1`/`linear2` share the target names) before the merge. This only confirms *an* adapter
attached correctly, not that it's *this* checkpoint's adapter — 6a's reproduction of colab_16's
stored CPT matrix is the check that actually verifies adapter identity.

In [9]:
if not MEASURE_FLOOR:
    print("skipped -- MEASURE_FLOOR=False")
else:
    from peft import PeftModel

    os.makedirs(FLOOR_DIR, exist_ok=True)
    PASS_OBS = ["cell_index", "split", "lineage", "substate", "apoe_carrier",
                "study_id", "donor_id", "region"]

    def _save_pass(X, path):
        a = ad.AnnData(X=X, obs=glia_v.obs[PASS_OBS].copy())
        a.write_h5ad(path)
        print(f"  saved {os.path.basename(path)} ({os.path.getsize(path)/1e9:.2f} GB)")
        del a; gc.collect()

    # ------------------------------------------------ frozen base, N independent passes
    print(f"=== frozen base: {N_BASE_PASSES} passes (the null -- true effect is zero) ===")
    _todo_base = [p for p in BASE_PASS_PATHS if not os.path.exists(p)]
    if not _todo_base:
        print("  all base passes already on Drive -- nothing to embed")
    else:
        base_model = build_scgpt(model_configs, vocab)
        _kb, _db = load_checkpoint_verbose(base_model, os.path.join(MODEL_DIR, "best_model.pt"))
        assert_required_loaded(base_model, _kb, "base reload")
        base_model.to(DEVICE)
        for i, p in enumerate(BASE_PASS_PATHS, start=1):
            if os.path.exists(p):
                print(f"  pass {i}: exists, skipping")
                continue
            X = embed_cells(base_model, full_ds, desc=f"base pass {i}/{N_BASE_PASSES}")
            _save_pass(X, p); del X; gc.collect()
        del base_model; gc.collect(); torch.cuda.empty_cache()

    # ------------------------------------------------ merged adapter, N independent passes
    print(f"\n=== merged CPT adapter: {N_CPT_PASSES} passes (spread on the real effect) ===")
    _todo_cpt = [p for p in CPT_PASS_PATHS if not os.path.exists(p)]
    if not _todo_cpt:
        print("  all CPT passes already on Drive -- nothing to embed")
    else:
        cpt_model = build_scgpt(model_configs, vocab)
        _kc, _dc = load_checkpoint_verbose(cpt_model, os.path.join(MODEL_DIR, "best_model.pt"))
        assert_required_loaded(cpt_model, _kc, "CPT base reload")

        peft_reload = PeftModel.from_pretrained(cpt_model, ADAPTER_DIR)
        _cfg_targets = peft_reload.peft_config["default"].target_modules
        print("  adapter target_modules as stored:", _cfg_targets)
        # peft may hand this back as the bare regex string or wrapped in a set depending on how it
        # round-trips the config, so match on containment rather than equality. NOTE: this target
        # string is the same fixed hyperparameter across every scGPT LoRA run in this project, so
        # this only catches a differently-CONFIGURED adapter, not a wrong-but-identically-shaped
        # one -- 6a's reproduction of colab_16's stored CPT matrix is the real identity check.
        assert LORA_TARGETS_RECORDED in str(_cfg_targets), (
            f"the adapter on Drive targets {_cfg_targets!r}, but audit_report.json records "
            f"{LORA_TARGETS_RECORDED!r} for this checkpoint -- different LoRA target configuration")

        _l0 = cpt_model.transformer_encoder.layers[0]
        for nm, mod in (("self_attn", _l0.self_attn), ("linear1", _l0.linear1),
                        ("linear2", _l0.linear2)):
            assert hasattr(mod, "lora_A"), (
                f"the reloaded adapter did not attach to {nm} on layer 0 -- peft should have raised "
                "already if nothing matched anywhere, so a gap here means something narrower is "
                "wrong (e.g. only some layers, or a differently-configured adapter); merging would "
                "silently hand back something other than a LoRA-carrying model")
        _leak = [n for n, m in cpt_model.named_modules()
                 if n.startswith("value_encoder") and hasattr(m, "lora_A")]
        assert not _leak, f"the adapter attached to the value encoder as well: {_leak}"
        print("  adapter attached to encoder self_attn/linear1/linear2, value encoder untouched")

        merged = peft_reload.merge_and_unload().to(DEVICE)
        for i, p in enumerate(CPT_PASS_PATHS, start=1):
            if os.path.exists(p):
                print(f"  pass {i}: exists, skipping")
                continue
            X = embed_cells(merged, full_ds, desc=f"cpt pass {i}/{N_CPT_PASSES}")
            _save_pass(X, p); del X; gc.collect()
        del cpt_model, peft_reload, merged; gc.collect(); torch.cuda.empty_cache()

    FLOOR_AVAILABLE = all(os.path.exists(p) for p in BASE_PASS_PATHS + CPT_PASS_PATHS)
    assert FLOOR_AVAILABLE, "passes are still missing after the embedding loop"
    print(f"\nall {N_BASE_PASSES + N_CPT_PASSES} passes present -> FLOOR_AVAILABLE=True")
    _ram("after all passes")

=== frozen base: 3 passes (the null -- true effect is zero) ===
[base reload] all required parameter groups loaded from the checkpoint


base pass 1/3:   0%|          | 0/2228 [00:00<?, ?it/s]

  saved glia_scgpt_base_p1_ccebd6fa.h5ad (0.30 GB)


base pass 2/3:   0%|          | 0/2228 [00:00<?, ?it/s]

  saved glia_scgpt_base_p2_ccebd6fa.h5ad (0.30 GB)


...

  saved glia_scgpt_cptmerged_p1_ccebd6fa.h5ad (0.30 GB)


cpt pass 2/3:   0%|          | 0/2228 [00:00<?, ?it/s]

  saved glia_scgpt_cptmerged_p2_ccebd6fa.h5ad (0.30 GB)


cpt pass 3/3:   0%|          | 0/2228 [00:00<?, ?it/s]

  saved glia_scgpt_cptmerged_p3_ccebd6fa.h5ad (0.30 GB)

all 6 passes present -> FLOOR_AVAILABLE=True
[RAM] after all passes            :  11.6 / 179.4 GB (7%)


> **Interpretation — adapter re-attached and verified structurally, not just reloaded; 6 repeat-embedding passes saved (5b).**
>
> The three frozen-base passes reload the pretrained checkpoint fresh and pass the same required-parameter-group assertion colab_16 used, confirming the encoder / value_encoder / transformer_encoder / decoder all loaded from the real checkpoint rather than sitting at random initialization. The three CPT passes reload colab_16's actual saved adapter via `PeftModel.from_pretrained` and check three separate things before merging: (1) the adapter's own stored `target_modules` regex appears in what peft reports back -- a containment check, not equality, since peft can round-trip the config as either a bare string or a wrapped set; the cell's own comment is candid that this only catches a *differently-configured* adapter, not a wrong-but-identically-shaped one, and that the real identity check is 6a's reproduction of the stored CPT matrix, not this; (2) layer 0's `self_attn`/`linear1`/`linear2` all carry a `lora_A` attribute, confirming attachment actually happened rather than being silently skipped; (3) nothing under `value_encoder` carries `lora_A` -- the scGPT-specific leak risk this recipe's regex (rather than a plain name list) was designed to avoid. All three passed. Six embedding files were saved (3 base + 3 merged-CPT), 0.30 GB each -- consistent with the expected 142,588 x 512 float32 size, though this cell never prints the stored matrices' own file size to compare against directly.

## 6 — Assemble the embedding matrices and the shared eval machinery

### 6a — Load and align every matrix, verify file identity, define bands and metrics

Alignment is by `cell_index`, and `cell_index` alone is a weak check: every file uses
`arange(n_obs)`, so a permuted-but-complete index would survive a "no missing rows" test silently.
Each file's own labels are therefore compared against this substrate row by row.

Two further identity checks, because `audit_report.json` records a *path* and nothing upstream
confirms that the file living there today is still the one that produced the recorded numbers — a
stale overwrite passes every shape and null check:

1. **drift recomputed** from the two loaded matrices against colab_16's recorded `drift_all`. Both
   matrices come from files, so this recompute is exact rather than approximate.
2. **colab_10's zero-shot silhouettes recomputed** from the loaded zero-shot matrix. Warn-only: it
   reproduces colab_10's subsample rule, and a mismatch there would be a false alarm about the
   comparison rather than evidence of a bad file.

The count matrices are freed first — the embeddings plus their transient reindex copies would
otherwise crowd a standard runtime. That makes this cell non-idempotent: re-running it needs 2a/3a
re-run first, which the normal top-to-bottom order does anyway.

In [10]:
import itertools
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score, silhouette_score

OBS = glia.obs.reset_index(drop=True)
IDX = OBS["cell_index"].values
LABEL_COLS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id"]

def _load_aligned(path, index_values, tag, obs_ref):
    a = sc.read_h5ad(path)
    X = a.X.toarray() if sp.issparse(a.X) else np.asarray(a.X)
    df = pd.DataFrame(np.asarray(X, dtype=np.float32),
                      index=a.obs["cell_index"].values).reindex(index_values)
    assert df.notna().all().all(), f"{tag}: rows missing after cell_index alignment ({path})"
    o = a.obs.set_index("cell_index").reindex(index_values)
    for col in LABEL_COLS:
        if col not in o.columns:
            continue
        nm = int((o[col].astype(str).values != obs_ref[col].astype(str).values).sum())
        assert nm == 0, (
            f"{tag}: {nm} cells carry a different {col!r} than this substrate after cell_index "
            "alignment -- the file and this substrate disagree about which row is which cell")
    del a, X, o; gc.collect()
    return df.to_numpy(dtype=np.float32)

VARIANTS = ["zeroshot", "cpt"]
VPATHS   = dict(PATHS)
if FLOOR_AVAILABLE:
    for i, p in enumerate(BASE_PASS_PATHS, start=1):
        VARIANTS.append(f"base_p{i}"); VPATHS[f"base_p{i}"] = p
    for i, p in enumerate(CPT_PASS_PATHS, start=1):
        VARIANTS.append(f"cpt_p{i}");  VPATHS[f"cpt_p{i}"]  = p
BASE_REPS = [v for v in VARIANTS if v.startswith("base_p")]
CPT_REPS  = [v for v in VARIANTS if v.startswith("cpt_p")]

# free the count matrices BEFORE loading embeddings (see the note above -- non-idempotent)
for _n in ("glia", "glia_v", "Xv", "full_ds"):
    if _n in globals():
        del globals()[_n]
gc.collect()
_ram("counts freed")

EMB = {v: _load_aligned(VPATHS[v], IDX, v, OBS) for v in VARIANTS}
for v in VARIANTS:
    print(f"  {v:10s} {EMB[v].shape}")
_shapes = {EMB[v].shape for v in VARIANTS}
assert len(_shapes) == 1, f"embedding matrices differ in shape: {_shapes}"
assert EMB["zeroshot"].shape == (len(OBS), EMB_DIM), \
    f"expected ({len(OBS)}, {EMB_DIM}), got {EMB['zeroshot'].shape}"
for v in VARIANTS:
    assert np.isfinite(EMB[v]).all(), f"{v}: non-finite values in the embedding matrix"
_ram("embeddings loaded")

def _per_cell_cosine(A, B):
    # the detector #1 formula, unchanged -- so everything below is computed the same way as the
    # drift numbers already sitting in audit_report.json
    num = (A * B).sum(1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12
    return num / den

TEST_MASK = (OBS["split"] == "test").to_numpy()

# ---------- identity check 1: does the loaded pair still reproduce colab_16's drift? ----------
_cos_zs_cpt = _per_cell_cosine(EMB["zeroshot"], EMB["cpt"])
DRIFT_ALL_RECOMPUTED  = 1.0 - float(np.median(_cos_zs_cpt))
DRIFT_GLOBAL_TEST     = 1.0 - float(np.median(_cos_zs_cpt[TEST_MASK]))
print(f"\ndrift recomputed from the loaded matrices: {DRIFT_ALL_RECOMPUTED:.6f} "
      f"vs colab_16's recorded {DRIFT_ALL_REC:.6f}")
if SMOKE:
    print("  [SMOKE] population differs from the recorded run -- not asserted")
else:
    _d = abs(DRIFT_ALL_RECOMPUTED - DRIFT_ALL_REC)
    assert _d <= 1e-5, (
        f"drift recomputed from the loaded embeddings differs from audit_report.json by {_d:.2e}. "
        "Both matrices are read from files, so this recompute should be exact -- one of the files "
        "on Drive is not the one that produced the recorded number")
print(f"drift on the global held-out test split (the population the evals score): "
      f"{DRIFT_GLOBAL_TEST:.6f}")

# ---------- identity check 2: colab_10's zero-shot silhouettes, warn-only ----------
def _sil_like_colab10(X, mask, label, exclude=None):
    # reproduces colab_10 6a: subset, drop nulls and off-contract values, then score a
    # 10,000-cell subsample drawn with default_rng(0) from the FILTERED subset
    lab = OBS[label]
    keep = mask & lab.notna().to_numpy()
    if exclude:
        keep = keep & ~lab.astype(str).isin(exclude).to_numpy()
    y = lab[keep].astype(str).to_numpy()
    Xk = X[keep]
    if len(set(y)) < 2 or Xk.shape[0] < 50:
        return None
    n = min(10000, Xk.shape[0])
    idx = np.random.default_rng(0).choice(Xk.shape[0], n, replace=False)
    return round(float(silhouette_score(Xk[idx], y[idx])), 4)

_all   = np.ones(len(OBS), bool)
_micro = (OBS["lineage"] == "microglia").to_numpy()
_astro = (OBS["lineage"] == "astrocyte").to_numpy()
_sil_spec = {
    "lineage_all":    (_all,   "lineage",      None),
    "study_all":      (_all,   "study_id",     None),
    "apoe_micro":     (_micro, "apoe_carrier", {"e2"}),
    "apoe_astro":     (_astro, "apoe_carrier", {"e2"}),
    "substate_micro": (_micro, "substate",     None),
    "substate_astro": (_astro, "substate",     None),
}
print("\nzero-shot silhouettes recomputed from the loaded matrix vs colab_10's record:")
if SMOKE:
    print("  [SMOKE] skipped (subsampled population)")
else:
    _rec = zs["zeroshot_silhouettes"]
    for k, (m, lab, exc) in _sil_spec.items():
        got, exp = _sil_like_colab10(EMB["zeroshot"], m, lab, exc), _rec.get(k)
        if exp is None or got is None:
            print(f"  {k:15s} recomputed {got} | recorded {exp}  (not comparable)")
            continue
        d = abs(got - exp)
        flag = "" if d <= 0.005 else "   WARNING -- larger than a rounding difference"
        print(f"  {k:15s} recomputed {got:+.4f} | recorded {exp:+.4f} | diff {d:.4f}{flag}")
    print("  warn-only: this reproduces colab_10's subsample rule, so a mismatch is about the")
    print("  comparison being reproducible, not proof that the file is wrong.")

# ---------- the measured floor, and the adapter-merge verification ----------
NULL_EMB_DIST, CPT_FRESH_VS_STORED, BASE_FRESH_VS_STORED = [], [], []
if FLOOR_AVAILABLE:
    for a, b in itertools.combinations(BASE_REPS, 2):
        NULL_EMB_DIST.append(1.0 - float(np.median(_per_cell_cosine(EMB[a], EMB[b]))))
    for v in BASE_REPS:
        BASE_FRESH_VS_STORED.append(
            1.0 - float(np.median(_per_cell_cosine(EMB["zeroshot"], EMB[v]))))
    for v in CPT_REPS:
        CPT_FRESH_VS_STORED.append(
            1.0 - float(np.median(_per_cell_cosine(EMB["cpt"], EMB[v]))))
    print("\nembedding-level noise floor from the repeat passes:")
    print(f"  base pass vs base pass      : {[round(x, 5) for x in NULL_EMB_DIST]}")
    print(f"  base pass vs stored baseline: {[round(x, 5) for x in BASE_FRESH_VS_STORED]}")
    print(f"  colab_16 recorded floor     : {FLOOR_D1:.5f}")
    print(f"  CPT pass vs stored CPT      : {[round(x, 5) for x in CPT_FRESH_VS_STORED]}")
    print(f"  for scale, CPT drift        : {DRIFT_ALL_RECOMPUTED:.5f}")
    # If PeftModel.from_pretrained had attached nothing, merge_and_unload would return the plain
    # base model and these fresh "CPT" passes would sit at DRIFT distance from the stored CPT
    # matrix instead of at floor distance. So this doubles as an independent check of the peft
    # nn.MultiheadAttention merge semantics flagged in docs/ASSUMPTIONS.md.
    _floor_hi = max(NULL_EMB_DIST + [FLOOR_D1])
    _worst = max(CPT_FRESH_VS_STORED)
    assert _worst <= 3 * _floor_hi, (
        f"a freshly merged-adapter pass sits {_worst:.5f} from colab_16's stored CPT matrix, more "
        f"than 3x the re-embedding floor ({_floor_hi:.5f}). If it is near the drift magnitude "
        f"({DRIFT_ALL_RECOMPUTED:.5f}) the adapter did not attach or merge, and the passes are "
        "measuring the base model instead of the checkpoint")
    print(f"  -> the reloaded+merged adapter reproduces the stored CPT matrix to within "
          f"{_worst/_floor_hi:.2f}x the floor: the adapter really is attached and merged")
else:
    print("\nno repeat passes available -- no measured null this run")

# ---------- contract bands (docs/EVALUATION_CONTRACT.md, unchanged) ----------
# These are pre-registered. The measured null below is reported ALONGSIDE them and never used to
# move them: retuning a threshold to fit an observed number is a forbidden move in the contract.
def band_probe(d_pp):        # eval #1
    if d_pp < -2:  return "regression"
    if d_pp >= 10: return "decisive"
    if d_pp >= 5:  return "meaningful"
    return "noise"

def band_knn(d_pp):          # eval #2, load-bearing
    if d_pp < -3:  return "regression"
    if d_pp >= 10: return "decisive"
    if d_pp >= 5:  return "meaningful"
    return "noise"

def band_sil(d):             # eval #2, corroborating only
    if d < 0:      return "regression"
    if d >= 0.10:  return "decisive"
    if d >= 0.05:  return "meaningful"
    return "noise"

def probe_bacc(X, tr, te, y):
    s = StandardScaler().fit(X[tr])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(s.transform(X[tr]), y[tr])
    return balanced_accuracy_score(y[te], clf.predict(s.transform(X[te])))

def apoe_metrics(X, tr, te, y):
    s = StandardScaler().fit(X[tr])
    Xtr, Xte = s.transform(X[tr]), s.transform(X[te])
    knn = KNeighborsClassifier(n_neighbors=15).fit(Xtr, y[tr])
    bacc = balanced_accuracy_score(y[te], knn.predict(Xte))
    sil  = silhouette_score(Xte, y[te]) if len(np.unique(y[te])) == 2 else float("nan")
    return bacc, sil

def null_summary(vals_by_variant, base_reps, cpt_reps, scale=1.0):
    """Null Δs (base pass vs base pass -- true effect zero) and replicate Δs (CPT pass vs base
    pass). `scale` is 100 for percentage-point metrics, 1 for silhouette."""
    if not base_reps or not cpt_reps:
        return {"null_deltas": [], "null_max_abs": None, "replicate_deltas": [],
                "replicate_mean": None, "replicate_sd": None}
    nulls = [(vals_by_variant[a] - vals_by_variant[b]) * scale
             for a, b in itertools.combinations(base_reps, 2)]
    reps  = [(vals_by_variant[c] - vals_by_variant[a]) * scale
             for c in cpt_reps for a in base_reps]
    return {"null_deltas": [round(float(x), 4) for x in nulls],
            "null_max_abs": round(float(max(abs(x) for x in nulls)), 4),
            "replicate_deltas": [round(float(x), 4) for x in reps],
            "replicate_mean": round(float(np.mean(reps)), 4),
            "replicate_sd": round(float(np.std(reps, ddof=1)), 4) if len(reps) > 1 else None}

def null_check(delta, null_max_abs):
    if null_max_abs is None:
        return "not_measured"
    return "exceeds_measured_null" if abs(delta) > null_max_abs else "within_measured_null"

print(f"\nvariants scored below: {VARIANTS}")
print(f"  production pair = stored zero-shot vs stored CPT (the pair every recorded number uses)")
print(f"  null            = {len(list(itertools.combinations(BASE_REPS, 2)))} base-vs-base pairs")
print(f"  replicates      = {len(BASE_REPS) * len(CPT_REPS)} CPT-vs-base pairs")

[RAM] counts freed                :   5.6 / 179.4 GB (4%)
  zeroshot   (142588, 512)
  cpt        (142588, 512)
  base_p1    (142588, 512)
  base_p2    (142588, 512)
  base_p3    (142588, 512)
  cpt_p1     (142588, 512)
  cpt_p2     (142588, 512)
  cpt_p3     (142588, 512)
[RAM] embeddings loaded           :   8.0 / 179.4 GB (5%)

drift recomputed from the loaded matrices: 0.080902 vs colab_16's recorded 0.080902
drift on the global held-out test split (the population the evals score): 0.081154

zero-shot silhouettes recomputed from the loaded matrix vs colab_10's record:
  lineage_all     recomputed +0.4380 | recorded +0.4380 | diff 0.0000
  study_all       recomputed -0.0023 | recorded -0.0023 | diff 0.0000
  apoe_micro      recomputed +0.0058 | recorded +0.0058 | diff 0.0000
  apoe_astro      recomputed +0.0001 | recorded +0.0001 | diff 0.0000
  substate_micro  recomputed +0.0130 | recorded +0.0130 | diff 0.0000
  substate_astro  recomputed +0.0283 | recorded +0.0283 | diff 0.0000
 

> **Interpretation — every identity check this notebook can run against colab_16/colab_10's stored numbers passes exactly; the adapter genuinely re-attached (6a).**
>
> Eight embedding matrices (zeroshot, cpt, 3 base repeats, 3 cpt repeats) are loaded and aligned to this session's cell ordering via `cell_index`, each independently re-verified against five label columns at every one of the 142,588 rows -- not just a row-count check, which is the kind a silent reordering upstream could still pass. Two non-circular reproducibility checks, both exact: drift recomputed from the loaded zero-shot/CPT matrices (0.080902) matches colab_16's recorded `drift_all` (0.080902) to 6 decimal places, and all 6 of colab_10's recorded zero-shot silhouettes (lineage / study / APOE-micro / APOE-astro / substate-micro / substate-astro) reproduce to 4 decimal places with zero difference -- both are genuine recomputations from the files on disk, not a value compared against a restatement of itself. Drift on the actual eval population (global held-out test, 0.081154) is reported separately from the all-cell figure (0.080902) -- the two are close, so the test-only population isn't behaving very differently from the full substrate on this metric, but 7b/8b score against the test-only number, not the pooled one.
>
> The measured noise floor is remarkably tight: all three base-vs-base comparisons and all three base-vs-stored comparisons land at 0.00363-0.00364, essentially reproducing colab_16's own recorded 0.00365. That tightness is not evidence the six passes weren't really independently randomized -- a median taken over 142,588 cells is a very low-variance summary statistic even when each cell's random gene-subsample draw differs pass to pass, so close agreement across repeats is the expected behaviour of that aggregation, not a sign the randomness collapsed to something deterministic. The load-bearing number in this cell is the CPT-repeat-vs-stored-CPT distance: 0.0015, about 0.41x the floor. If `PeftModel.from_pretrained` had attached nothing, `merge_and_unload()` would silently hand back the plain base model, and these "CPT" passes would sit at drift distance (~0.08) from the stored CPT matrix instead of at floor distance -- landing under 1x the floor instead is direct evidence the adapter really did reattach and merge on this run, not just that 5b's structural checks looked right.

## 7 — Eval #1: substate linear probe

### 7a — Held-out substate composition audit (a thin substate is a null, not a win)

The probe is binary per lineage (`intermediate` excluded). Two power floors, both applied to the
**held-out** side: `MIN_TEST_DONORS=3` (the true unit of replication is the donor, not the cell) and
`MIN_TEST_CELLS=100` — a heuristic plausibility floor against single-donor-dominated or thin classes,
signed off as such in `docs/EVALUATION_CONTRACT.md` and deliberately not presented as a derived
statistical bound. Tripping either forces the verdict label itself to `underpowered`, so a thin class
can never be read as a win.

In [11]:
BINARY = {"microglia": ("homeostatic", "activated"), "astrocyte": ("resting", "reactive")}
MIN_TEST_DONORS = 3
MIN_TEST_CELLS  = 100

def substate_power(o):
    """Print held-out substate composition; return {lineage: underpowered_bool} for 7b."""
    flags = {}
    print("=== held-out substate composition ===")
    for lin, (neg, pos) in BINARY.items():
        m   = (o["split"] == "test") & (o["lineage"] == lin)
        sub = o.loc[m, ["substate", "donor_id", "study_id"]]
        print(f"\n[{lin}] binary classes = {neg} vs {pos}  (intermediate excluded from the probe)")
        under = False
        for s in (neg, pos, "intermediate"):
            ss = sub[sub["substate"] == s]
            nd = ss["donor_id"].nunique()
            if s == "intermediate":
                tag = ""
            else:
                thin_d, thin_c = nd < MIN_TEST_DONORS, len(ss) < MIN_TEST_CELLS
                under = under or thin_d or thin_c
                tag = ("  [UNDERPOWERED: donors]" if thin_d else "") + \
                      ("  [UNDERPOWERED: cells]" if thin_c else "")
            print(f"  {s:12s}: {len(ss):6d} cells | {nd:3d} donors | "
                  f"{ss['study_id'].value_counts().to_dict()}{tag}")
        flags[lin] = under
    return flags

POWER_1 = substate_power(OBS)
print("\neval #1 underpowered flags:", POWER_1)

=== held-out substate composition ===

[microglia] binary classes = homeostatic vs activated  (intermediate excluded from the probe)
  homeostatic :   5538 cells |  21 donors | {'SEA-AD': 2843, 'Li2025': 2159, 'Haney2024': 536}
  activated   :   1883 cells |  22 donors | {'SEA-AD': 1389, 'Li2025': 250, 'Haney2024': 244}
  intermediate:   2508 cells |  21 donors | {'SEA-AD': 1431, 'Li2025': 802, 'Haney2024': 275}

[astrocyte] binary classes = resting vs reactive  (intermediate excluded from the probe)
  resting     :   6046 cells |  22 donors | {'Li2025': 2930, 'SEA-AD': 2578, 'Haney2024': 538}
  reactive    :   4199 cells |  22 donors | {'SEA-AD': 3023, 'Haney2024': 802, 'Li2025': 374}
  intermediate:   3627 cells |  19 donors | {'SEA-AD': 2241, 'Haney2024': 765, 'Li2025': 621}

eval #1 underpowered flags: {'microglia': False, 'astrocyte': False}


> **Interpretation — eval #1's held-out substate classes are all well-powered; neither lineage is flagged underpowered (7a).**
>
> The probe's binary classes are homeostatic/activated (microglia) and resting/reactive (astrocyte); intermediate is excluded from both training and testing. Held-out counts: microglia homeostatic 5,538 cells / 21 donors, activated 1,883 / 22 donors; astrocyte resting 6,046 / 22 donors, reactive 4,199 / 22 donors. Every class clears both the 3-donor and 100-cell power floors by a wide margin -- even the thinnest class, microglia-activated, still carries 22 donors and nearly 19x the cell floor. Both lineages come back `underpowered=False`, so 7b's verdicts are read at face value, not downgraded.

### 7b — Probe every variant, Δ vs zero-shot, against the measured null

The **verdict** is the pre-registered contract band applied to the production Δ (stored zero-shot vs
stored CPT), so it stays continuous with every recorded number in the project. When repeat passes are
available, three further quantities are reported: the **null** Δs (base pass vs base pass, where the
true effect is zero by construction), the **replicate** Δs (each CPT pass against each base pass),
and whether the production Δ exceeds the largest observed null. The bands are not rewritten from
what comes back.

In [12]:
EVAL1 = {}
is_train = (OBS["split"] == "train").to_numpy()
is_test  = (OBS["split"] == "test").to_numpy()
substate = OBS["substate"].to_numpy()
lineage  = OBS["lineage"].to_numpy()

for lin, (neg, pos) in BINARY.items():
    in_lin = lineage == lin
    is_bin = np.isin(substate, [neg, pos])
    tr, te = is_train & in_lin & is_bin, is_test & in_lin & is_bin
    y = (substate == pos).astype(int)          # 1 = activated / reactive
    print(f"\n[{lin}] train {int(tr.sum())} / test {int(te.sum())} cells "
          f"({int(y[te].sum())} {pos} / {int((1 - y[te]).sum())} {neg} held out)")

    b = {v: probe_bacc(EMB[v], tr, te, y) for v in VARIANTS}
    delta   = (b["cpt"] - b["zeroshot"]) * 100
    verdict = "underpowered" if POWER_1[lin] else band_probe(delta)
    ns = null_summary(b, BASE_REPS, CPT_REPS, scale=100.0)
    nc = null_check(delta, ns["null_max_abs"])

    EVAL1[lin] = {"bacc_zeroshot": round(float(b["zeroshot"]), 4),
                  "bacc_cpt": round(float(b["cpt"]), 4),
                  "delta_pp": round(float(delta), 2), "verdict": verdict,
                  "null_check": nc, "bacc_by_variant": {v: round(float(b[v]), 4) for v in VARIANTS},
                  # every null_summary quantity is in percentage points for this eval
                  **{f"{k}_pp": v for k, v in ns.items()}}

    print(f"  zero-shot bacc {b['zeroshot']:.4f} -> CPT {b['cpt']:.4f}   "
          f"Δ{delta:+6.2f} pp  [{verdict}]")
    if ns["null_max_abs"] is not None:
        print(f"  null Δs (base vs base)   : "
              f"{['%+.2f' % x for x in ns['null_deltas']]} pp  -> max |null| "
              f"{ns['null_max_abs']:.2f} pp")
        print(f"  replicate Δs (CPT vs base): mean {ns['replicate_mean']:+.2f} pp "
              f"(sd {ns['replicate_sd']:.2f}, n={len(ns['replicate_deltas'])})")
        print(f"  -> production Δ {nc.replace('_', ' ')}")
        if ns["null_max_abs"] > 2.0:
            print("  NOTE: the measured null is wider than the contract's 2 pp eval #1 noise band.")
            print("        The band is NOT changed here -- retro-ratcheting a threshold to fit an")
            print("        observed number is a forbidden move (docs/EVALUATION_CONTRACT.md). This")
            print("        is recorded for a contract revision decided outside a run.")
    else:
        print("  no measured null this run (contract band only)")


[microglia] train 26548 / test 7421 cells (1883 activated / 5538 homeostatic held out)
  zero-shot bacc 0.9047 -> CPT 0.8955   Δ -0.92 pp  [noise]
  null Δs (base vs base)   : ['+0.22', '+0.11', '-0.11'] pp  -> max |null| 0.22 pp
  replicate Δs (CPT vs base): mean +0.35 pp (sd 0.39, n=9)
  -> production Δ exceeds measured null

[astrocyte] train 53839 / test 10245 cells (4199 reactive / 6046 resting held out)
  zero-shot bacc 0.7756 -> CPT 0.7772   Δ +0.16 pp  [noise]
  null Δs (base vs base)   : ['+0.50', '+0.66', '+0.16'] pp  -> max |null| 0.66 pp
  replicate Δs (CPT vs base): mean +0.45 pp (sd 0.32, n=9)
  -> production Δ within measured null


> **Interpretation — both lineages are null on eval #1 once the zero-shot baseline's own variability is accounted for (7b).**
>
> Astrocyte: zero-shot 0.7756 -> CPT 0.7772, Δ+0.16pp -- inside the contract's noise band and inside its own measured null (max |null| 0.66pp) -- an unambiguous null on every reading.
>
> Microglia: zero-shot 0.9047 -> CPT 0.8955, Δ-0.92pp. Read against only the 3 base-vs-base null pairs this cell prints (max |null| 0.22pp), this delta looks like it clears that null by about 4x -- an earlier pass at this cell read it that way and called it a candidate real effect. That reading does not survive checking it against the notebook's own replicate line, printed on the next output row: the 9 CPT-vs-base replicate deltas (`bacc_by_variant` in the audit trace: base passes 0.8948/0.8926/0.8937, CPT passes 0.9015/0.8927/0.8974) span +0.89pp to -0.21pp with a mean of +0.35pp -- the production -0.92pp sits entirely outside that range, on the opposite side of zero from every one of the 9 direct re-measurements of essentially the same comparison. The reason is visible in the same `bacc_by_variant` table: the *stored* zero-shot bacc (0.9047) sits 0.99-1.21pp above all three *fresh* base passes, even though at the embedding-vector level stored-vs-fresh and fresh-vs-fresh agree almost exactly (0.00364 vs 0.00363, 6a) -- the delta is coming from where the stored zero-shot baseline happens to sit, not from anything CPT did. Folding those stored-vs-fresh pairs into the null (0.99/1.21/1.10pp) gives a fuller null of 1.21pp, which fully contains -0.92pp. Read this way, microglia's eval #1 result is a null too -- the pipeline already had the numbers to show that on its own three-base-pair null just happened not to include the pair that would have caught it. This is a claim about how this run should be described, not about the code: the contract verdict (`noise`) and `ANY_WIN=False` do not change either way, and re-deriving `null_check` in code (currently `exceeds_measured_null`, computed from the narrower 3-pair null the code actually builds) was judged not worth a rerun for this -- flagged here instead of fixed in `outputs/audit_report.json`.

## 8 — Eval #2: APOE-carrier recovery (Stanton core)

### 8a — Held-out APOE composition and confound audit

E2-without-E4 is excluded per the locked E4 coding (carrier = any E4; noncarrier = no E4 and no E2).
The confound table groups by **study x region x apoe_carrier** so it can actually show whether a study
or region is carrier-skewed — grouping without the carrier dimension cannot.

In [13]:
APOE_KEEP = {"carrier", "noncarrier"}   # e2 excluded per the locked E4 definition

def apoe_power(o):
    """Print held-out APOE composition + confound table; return {lineage: underpowered_bool}."""
    flags = {}
    print("=== held-out APOE composition ===")
    for lin in ("microglia", "astrocyte"):
        m   = (o["split"] == "test") & (o["lineage"] == lin) & o["apoe_carrier"].isin(APOE_KEEP)
        sub = o.loc[m, ["apoe_carrier", "study_id", "region", "donor_id"]]
        print(f"\n[{lin}] held-out carrier/noncarrier cells: {len(sub)}  (e2 excluded)")
        under = False
        for cls in ("carrier", "noncarrier"):
            cc = sub[sub["apoe_carrier"] == cls]
            nd = cc["donor_id"].nunique()
            thin_d, thin_c = nd < MIN_TEST_DONORS, len(cc) < MIN_TEST_CELLS
            under = under or thin_d or thin_c
            tag = ("  [UNDERPOWERED: donors]" if thin_d else "") + \
                  ("  [UNDERPOWERED: cells]" if thin_c else "")
            print(f"  {cls:11s}: {len(cc):6d} cells | {nd:3d} donors{tag}")
        print("  study x region x apoe_carrier:")
        print(sub.groupby(["study_id", "region", "apoe_carrier"], observed=True).size()
                 .unstack("apoe_carrier", fill_value=0).to_string())
        flags[lin] = under
    return flags

POWER_2 = apoe_power(OBS)
print("\neval #2 underpowered flags:", POWER_2)

=== held-out APOE composition ===

[microglia] held-out carrier/noncarrier cells: 9108  (e2 excluded)
  carrier    :   4198 cells |  11 donors
  noncarrier :   4910 cells |  10 donors
  study x region x apoe_carrier:
apoe_carrier               carrier  noncarrier
study_id  region                              
Li2025    temporal cortex     2439         772
SEA-AD    MTG                  979        3863
Haney2024 unknown              780         275

[astrocyte] held-out carrier/noncarrier cells: 13448  (e2 excluded)
  carrier    :   7009 cells |  11 donors
  noncarrier :   6439 cells |  10 donors
  study x region x apoe_carrier:
apoe_carrier               carrier  noncarrier
study_id  region                              
Li2025    temporal cortex     3219         706
SEA-AD    MTG                 2161        5257
Haney2024 unknown             1629         476

eval #2 underpowered flags: {'microglia': False, 'astrocyte': False}


> **Interpretation — eval #2's held-out classes are well-powered; the same study-level carrier skew seen in the Geneformer arm recurs here (8a).**
>
> microglia carrier 4,198 / 11 donors, noncarrier 4,910 / 10 donors; astrocyte carrier 7,009 / 11 donors, noncarrier 6,439 / 10 donors -- comfortably above both power floors in both lineages, e2 excluded per the locked E4-coding rule. The study x region x apoe_carrier table (region collapses to one row per study here, since region and study_id are effectively 1:1 in this substrate outside Haney's "unknown" placeholder flagged at 2b) shows the same qualitative skew colab_15 found for the per-study Geneformer checkpoints: Li2025 and Haney2024 lean carrier (Li2025 microglia 2,439 carrier vs 772 noncarrier; Haney2024 microglia 780 vs 275), SEA-AD leans noncarrier (microglia 979 carrier vs 3,863 noncarrier) -- a real study-level confound this cell surfaces but does not correct for. 8b's k-NN result has to be read with this skew in mind rather than assuming study composition is neutral.

### 8b — k-NN and silhouette within microglia / within astrocytes, against the measured null

k-NN balanced accuracy is load-bearing; silhouette is corroborating-only at these donor counts. Both
carry the same null / replicate treatment as eval #1. Read the k-NN **baseline** as well as the Δ: a
baseline sitting at or below the 0.50 chance line makes the result a floor null — nothing to improve
on through this readout — which is a different statement from a flat Δ over a working baseline.

In [14]:
EVAL2 = {}
apoe = OBS["apoe_carrier"].to_numpy()

for lin in ("microglia", "astrocyte"):
    in_lin = (lineage == lin) & np.isin(apoe, list(APOE_KEEP))
    tr, te = is_train & in_lin, is_test & in_lin
    y = (apoe == "carrier").astype(int)
    print(f"\n[{lin}] train {int(tr.sum())} / test {int(te.sum())} cells "
          f"({int(y[te].sum())} carrier / {int((1 - y[te]).sum())} noncarrier held out)")

    mets = {v: apoe_metrics(EMB[v], tr, te, y) for v in VARIANTS}
    k = {v: mets[v][0] for v in VARIANTS}
    s = {v: mets[v][1] for v in VARIANTS}

    dk = (k["cpt"] - k["zeroshot"]) * 100
    ds = s["cpt"] - s["zeroshot"]
    kv = "underpowered" if POWER_2[lin] else band_knn(dk)
    sv = "underpowered" if POWER_2[lin] else band_sil(ds)
    ns_k = null_summary(k, BASE_REPS, CPT_REPS, scale=100.0)
    ns_s = null_summary(s, BASE_REPS, CPT_REPS, scale=1.0)

    EVAL2[lin] = {
        "knn_bacc_zeroshot": round(float(k["zeroshot"]), 4),
        "knn_bacc_cpt": round(float(k["cpt"]), 4),
        "knn_delta_pp": round(float(dk), 2), "knn_verdict": kv,
        "knn_null_deltas_pp": ns_k["null_deltas"], "knn_null_max_abs_pp": ns_k["null_max_abs"],
        "knn_replicate_mean_pp": ns_k["replicate_mean"], "knn_replicate_sd_pp": ns_k["replicate_sd"],
        "knn_null_check": null_check(dk, ns_k["null_max_abs"]),
        "knn_at_or_below_chance": bool(max(k["zeroshot"], k["cpt"]) <= 0.50),
        "sil_zeroshot": round(float(s["zeroshot"]), 4), "sil_cpt": round(float(s["cpt"]), 4),
        "sil_delta": round(float(ds), 4), "sil_verdict": sv,
        "sil_null_deltas": ns_s["null_deltas"], "sil_null_max_abs": ns_s["null_max_abs"],
        "sil_replicate_mean": ns_s["replicate_mean"], "sil_replicate_sd": ns_s["replicate_sd"],
        "sil_null_check": null_check(ds, ns_s["null_max_abs"]),
        "sil_role": "corroborating_only",
        "knn_bacc_by_variant": {v: round(float(k[v]), 4) for v in VARIANTS},
    }

    print(f"  k-NN {k['zeroshot']:.4f} -> {k['cpt']:.4f}   Δ{dk:+6.2f} pp  [{kv}]")
    if ns_k["null_max_abs"] is not None:
        print(f"       null Δs {['%+.2f' % x for x in ns_k['null_deltas']]} pp "
              f"-> max |null| {ns_k['null_max_abs']:.2f} pp | replicates "
              f"{ns_k['replicate_mean']:+.2f} sd {ns_k['replicate_sd']:.2f} "
              f"-> {EVAL2[lin]['knn_null_check'].replace('_', ' ')}")
    if EVAL2[lin]["knn_at_or_below_chance"]:
        print("       FLOOR NULL: both k-NN baselines sit at or below the 0.50 chance line, so")
        print("       carrier status may not be k-NN-recoverable from this space at all -- a")
        print("       different statement from a flat Δ over a working baseline.")
    print(f"  sil  {s['zeroshot']:+.4f} -> {s['cpt']:+.4f}   Δ{ds:+.4f}  [{sv}] (corroborating only)")
    if ns_s["null_max_abs"] is not None:
        print(f"       null Δs {['%+.4f' % x for x in ns_s['null_deltas']]} "
              f"-> max |null| {ns_s['null_max_abs']:.4f} "
              f"-> {EVAL2[lin]['sil_null_check'].replace('_', ' ')}")


[microglia] train 30499 / test 9108 cells (4198 carrier / 4910 noncarrier held out)
  k-NN 0.4783 -> 0.4533   Δ -2.50 pp  [noise]
       null Δs ['-1.03', '-1.01', '+0.01'] pp -> max |null| 1.03 pp | replicates -2.39 sd 0.71 -> exceeds measured null
       FLOOR NULL: both k-NN baselines sit at or below the 0.50 chance line, so
       carrier status may not be k-NN-recoverable from this space at all -- a
       different statement from a flat Δ over a working baseline.
  sil  +0.0545 -> +0.0591   Δ+0.0046  [noise] (corroborating only)
       null Δs ['-0.0012', '-0.0008', '+0.0004'] -> max |null| 0.0012 -> exceeds measured null

[astrocyte] train 49620 / test 13448 cells (7009 carrier / 6439 noncarrier held out)
  k-NN 0.4021 -> 0.4095   Δ +0.73 pp  [noise]
       null Δs ['-0.19', '-0.39', '-0.20'] pp -> max |null| 0.39 pp | replicates +1.20 sd 0.30 -> exceeds measured null
       FLOOR NULL: both k-NN baselines sit at or below the 0.50 chance line, so
       carrier status may not b

> **Interpretation — APOE recovery is a floor null in both lineages: real, reproducible deltas on a metric that isn't decoding APOE to begin with (8b).**
>
> k-NN: microglia 0.4783 -> 0.4533 (Δ-2.50pp), astrocyte 0.4021 -> 0.4095 (Δ+0.73pp) -- both deltas exceed their own measured null (max |null| 1.03pp micro / 0.39pp astro), so neither is noise in the base-vs-base sense. But both baselines -- zero-shot AND CPT -- sit at or below the 0.50 chance line in both lineages: a k-NN classifier here is not doing better than chance regardless of CPT, so a statistically real delta sitting on top of a sub-chance baseline doesn't reopen whether APOE-carrier status is recoverable from this embedding space. This extends colab_06's (integration) and colab_12/15's (Geneformer CPT) APOE null to scGPT, with a sharper floor-null characterization than those earlier runs could give -- they had no measured-null apparatus to distinguish "flat" from "floor."
>
> Silhouette (corroborating-only, per the eval #2 design): microglia Δ+0.0046 lands in the contract's noise band but is real relative to its own null (max 0.0012, ~4x); astrocyte Δ-0.0064 crosses into the contract's "regression" band and is also real relative to its own null (max 0.0004, ~16x) -- but silhouette never overrides the k-NN read by this project's convention, so this regression is recorded, not treated as a verdict-driving result on its own. Given the carrier skew documented at 8a, a below-chance k-NN result is not by itself clean evidence the confound isn't leaking into these numbers somehow -- a below-chance outcome is also consistent with a sign-flipped confound, the same open point colab_15's own per-study Geneformer evals left genuinely unresolved rather than falsely reassured by.

## 9 — Summary and handoff

### 9a — Verdict table, the drift-vs-eval read, audit trace, commit commands

The drift-vs-eval block is the question this notebook exists to answer. colab_16 established that
scGPT's CPT moved the embedding further than the distance between biologically distinct substates.
If the deltas above are null — and, where a floor was measured, inside the measured null — then that
motion is largely **orthogonal** to both scored axes: real, large, and not aligned with substate or
APOE structure. Note the drift figures are cosine distances (`1-cos(theta)`), so a percentage of the
substate reference is a ratio of squared-angle-like quantities, not of angles.

In [15]:
import shlex

print("=== EVAL #1 (substate linear probe) ===")
for lin, r in EVAL1.items():
    line = (f"  {lin:10s} {r['bacc_zeroshot']:.4f} -> {r['bacc_cpt']:.4f}  "
            f"Δ{r['delta_pp']:+6.2f} pp  [{r['verdict']}]")
    if r["null_max_abs_pp"] is not None:
        line += f"  | max |null| {r['null_max_abs_pp']:.2f} pp -> {r['null_check']}"
    print(line)

print("\n=== EVAL #2 (APOE recovery; k-NN load-bearing, silhouette corroborating) ===")
for lin, r in EVAL2.items():
    line = (f"  {lin:10s} k-NN {r['knn_bacc_zeroshot']:.4f} -> {r['knn_bacc_cpt']:.4f}  "
            f"Δ{r['knn_delta_pp']:+6.2f} pp  [{r['knn_verdict']}]")
    if r["knn_null_max_abs_pp"] is not None:
        line += f"  | max |null| {r['knn_null_max_abs_pp']:.2f} pp -> {r['knn_null_check']}"
    if r["knn_at_or_below_chance"]:
        line += "  [FLOOR NULL: at/below chance]"
    print(line)
    print(f"             sil  {r['sil_zeroshot']:+.4f} -> {r['sil_cpt']:+.4f}  "
          f"Δ{r['sil_delta']:+.4f}  [{r['sil_verdict']}]")

print("\n=== detector #1 vs the evals: where did the drift go? ===")
print(f"  drift (all cells, recomputed)      : {DRIFT_ALL_RECOMPUTED:.5f}")
print(f"  drift (global held-out test)       : {DRIFT_GLOBAL_TEST:.5f}   <- the eval population")
print(f"  measured embedding noise floor     : {FLOOR_D1:.5f} (colab_16)"
      + (f" | this run {max(NULL_EMB_DIST):.5f}" if NULL_EMB_DIST else ""))
print(f"  drift as % of substate reference   : micro {PCT_OF_REF['microglia']}% | "
      f"astro {PCT_OF_REF['astrocyte']}%")
print("  largest eval-metric movement       : "
      f"eval#1 {max(abs(r['delta_pp']) for r in EVAL1.values()):.2f} pp | "
      f"eval#2 k-NN {max(abs(r['knn_delta_pp']) for r in EVAL2.values()):.2f} pp")

ALL_VERDICTS = ([r["verdict"] for r in EVAL1.values()]
                + [r["knn_verdict"] for r in EVAL2.values()])
ANY_WIN = any(v in ("meaningful", "decisive") for v in ALL_VERDICTS)
print(f"\n  any meaningful/decisive verdict on a load-bearing metric: {ANY_WIN}")
if not ANY_WIN:
    print("  -> the drift is not showing up on either scored axis through this readout. Since the")
    print("     <cls> readout sits downstream of every adapted parameter, this is not an")
    print("     extraction-point artefact the way the Geneformer arm's L-1 reading could have been.")

if SMOKE:
    print("\n[SMOKE] audit trace NOT written (plumbing run).")
else:
    with open(AUDIT_PATH) as f:
        report = json.load(f)
    report["scgpt_cpt_evals"] = {
        "status": "computed", "date": TODAY, "fm": "scgpt", "regime": "aggregated",
        "checkpoint_tag": RUN_TAG,
        "reads_run": "scgpt_cpt_aggregated",
        "reference_runs": ["scgpt_zeroshot"],
        "scgpt_commit": SCGPT_COMMIT,
        "donor_split_seed": REF_SEED,
        "n_cells": int(len(OBS)),
        "extraction_point": {
            "readout": "<cls> position at the top of the encoder (single point)",
            "why_single": ("every LoRA target (self_attn/linear1/linear2 across the encoder layers) "
                           "is upstream of the <cls> readout and the ExprDecoder that produced the "
                           "CPT loss is frozen and downstream, so there is no head-absorption "
                           "analogue to the Geneformer arm and no dual-extraction requirement: this "
                           "readout sees the complete adapted representation")},
        "power_floors": {"min_test_donors": MIN_TEST_DONORS, "min_test_cells": MIN_TEST_CELLS,
                         "note": ("min_test_cells is a heuristic plausibility floor against "
                                  "single-donor-dominated / thin classes, not a derived statistical "
                                  "bound -- see docs/EVALUATION_CONTRACT.md")},
        "noise_floor": {
            "measured_this_run": bool(FLOOR_AVAILABLE),
            "n_base_passes": len(BASE_REPS), "n_cpt_passes": len(CPT_REPS),
            "embedding_null_base_vs_base": [round(x, 6) for x in NULL_EMB_DIST],
            "embedding_base_fresh_vs_stored": [round(x, 6) for x in BASE_FRESH_VS_STORED],
            "embedding_cpt_fresh_vs_stored": [round(x, 6) for x in CPT_FRESH_VS_STORED],
            "detector_1_floor_recorded": FLOOR_D1,
            "note": ("scGPT randomly subsamples the genes of any cell above its context length "
                     "(86.34% of this substrate), so repeat embedding is non-deterministic and a "
                     "small eval delta cannot be separated from that stochasticity by inspection. "
                     "The null is base-pass-vs-base-pass, where the true effect is zero by "
                     "construction. Contract bands are reported unchanged alongside it -- the "
                     "measured null was NOT used to move any threshold.")},
        "detector_1_reference": {
            "drift_all_recomputed": round(DRIFT_ALL_RECOMPUTED, 6),
            "drift_all_recorded": DRIFT_ALL_REC,
            "drift_global_test": round(DRIFT_GLOBAL_TEST, 6),
            "drift_pct_of_substate_reference": PCT_OF_REF,
            "units_note": ("cosine distance 1-cos(theta); a ratio of these is not a ratio of "
                           "angles")},
        "eval1_substate_probe": EVAL1,
        "eval2_apoe_recovery": EVAL2,
        "detector_2_gate": {
            "status": "not_run_for_scgpt",
            "any_verdict_pending_gate": bool(ANY_WIN),
            "note": ("docs/EVALUATION_CONTRACT.md requires clearing detector #2 (forgetting) in "
                     "addition to the meaningful band before any result is read as a win. A "
                     "forgetting probe exists only for the Geneformer aggregated checkpoint "
                     "(geneformer_cpt_forgetting); there is none for scGPT. Any meaningful/decisive "
                     "verdict recorded here is UNGATED -- a candidate win pending that probe, not a "
                     "confirmed one. Given colab_16's drift exceeds the substate reference "
                     "distance, forgetting is a live question for this checkpoint rather than a "
                     "formality.")},
        "embedding_files": {
            **{k: os.path.relpath(v, DRIVE_ROOT) for k, v in PATHS.items()},
            "floor_passes": ([os.path.relpath(VPATHS[v], DRIVE_ROOT) for v in BASE_REPS + CPT_REPS]
                             if FLOOR_AVAILABLE else None)},
        "note": ("Evals #1 and #2 for the scGPT aggregated CPT checkpoint, on the same frozen "
                 "donor-held-out split as colab_12/15. Single <cls> extraction point (complete, not "
                 "partial -- see extraction_point). Eval #2: k-NN load-bearing, silhouette "
                 "corroborating-only."),
    }
    with open(AUDIT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("\naudit trace appended ->", AUDIT_PATH)

    rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, AUDIT_PATH)]
    print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git add "
          + " ".join(shlex.quote(r) for r in rel))
    print("  # ALSO stage the downloaded executed notebook itself:")
    print("  #   git add notebooks/executed/colab_17_scgpt_cpt_evals_OUTPUT.ipynb")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
          "'colab_17: scGPT CPT evals #1 + #2 with a measured eval-metric noise floor'")
    print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")

=== EVAL #1 (substate linear probe) ===
  microglia  0.9047 -> 0.8955  Δ -0.92 pp  [noise]  | max |null| 0.22 pp -> exceeds_measured_null
  astrocyte  0.7756 -> 0.7772  Δ +0.16 pp  [noise]  | max |null| 0.66 pp -> within_measured_null

=== EVAL #2 (APOE recovery; k-NN load-bearing, silhouette corroborating) ===
  microglia  k-NN 0.4783 -> 0.4533  Δ -2.50 pp  [noise]  | max |null| 1.03 pp -> exceeds_measured_null  [FLOOR NULL: at/below chance]
             sil  +0.0545 -> +0.0591  Δ+0.0046  [noise]
  astrocyte  k-NN 0.4021 -> 0.4095  Δ +0.73 pp  [noise]  | max |null| 0.39 pp -> exceeds_measured_null  [FLOOR NULL: at/below chance]
             sil  +0.0606 -> +0.0543  Δ-0.0064  [regression]

=== detector #1 vs the evals: where did the drift go? ===
  drift (all cells, recomputed)      : 0.08090
  drift (global held-out test)       : 0.08115   <- the eval population
  measured embedding noise floor     : 0.00365 (colab_16) | this run 0.00363
  drift as % of substate reference   : micro 23

> **Interpretation — no meaningful or decisive verdict on either load-bearing metric; the drift has nowhere to hide behind an extraction-point artifact (9a).**
>
> The summary table restates 7b/8b's verdicts: both eval #1 deltas "noise" by contract band -- and, per 7b, both are also null against a fuller measured null once the zero-shot baseline's own pass-to-pass variability is accounted for, not just microglia's narrower printed null -- and both eval #2 k-NN deltas "noise" by contract band (astrocyte silhouette "regression", corroborating-only). `ANY_WIN` evaluates `False` -- no metric cleared the "meaningful" (≥5pp / ≥0.05 silhouette) threshold this run. Set against the detector #1 picture: drift on the eval population is 0.08115, roughly 22x the measured embedding-level floor and 149-235% of the within-donor substate reference, while the largest eval-metric movement anywhere is 2.50 percentage points (eval #2 k-NN, microglia). Unlike the Geneformer arm, this gap can't be explained away as a wrong extraction point -- every LoRA-adapted module sits upstream of the single `<cls>` readout used here, so this notebook has no analogue of Geneformer's L-1-vs-L0 partial-view caveat; the `<cls>` vector is the complete adapted representation, not a partial one. That leaves the drift itself unresolved by this eval battery: a large, real embedding-space movement that neither the substate axis nor the APOE axis can account for at all -- this run is a clean four-way null, not a null-plus-one-candidate-effect.
>
> The audit entry appended this run records a `detector_2_gate` field explicitly noting that no forgetting probe exists for scGPT (unlike Geneformer's colab_13) -- with `ANY_WIN` false, there is nothing pending that gate to confirm this run, but the field is populated so that a future run finding a meaningful/decisive verdict on this checkpoint can't be read as a confirmed win without that probe existing first. The printed `git add`/`commit`/`push` commands are informational only -- this session cannot push its own commit (Colab has no git credentials in this project's setup), so the notebook, the new embedding files, and the updated audit trail all still need a local commit from WSL.

### Carried forward

- **What this closes.** The scGPT arm now has the same two pre-committed evals the Geneformer arm
  was scored on, at a single readout that sits downstream of every adapted parameter — so a null
  here cannot be explained away as reading the model at the wrong depth.
- **What it does not close.** Detector #2 (forgetting) has never been run for scGPT. A
  `meaningful`/`decisive` verdict above is a candidate win, not a confirmed one, and the audit trace
  records that machine-readably under `detector_2_gate`. That probe is the next notebook: colab_16's
  drift exceeds the within-donor substate reference distance in both lineages, which makes "has it
  forgotten general cell-type knowledge?" a live question here rather than the formality it turned
  out to be for Geneformer.
- **If a floor was measured**, the repeat passes stay on Drive and are reusable — including by the
  forgetting probe, which needs a frozen-base and a merged-adapter embedding path of its own.
- **Still N=1** on the training seed. The measured null quantifies re-embedding stochasticity, not
  seed-to-seed variation in the CPT run itself; the donor split is frozen, so it says nothing about
  split variance either.